# Recurrence of massive-activation channels across scenarios (PixArt-Sigma)

This code ranks the channels of the PixArt-Sigma image tokens by magnitude
in each scenario (one prompt and seed) and measures how often the same top
channels come back across scenarios.

The model is PixArt-Sigma-XL-2-1024-MS (28 blocks, hidden size 1152). The
grid is 20 prompts by 5 seeds, so 100 scenarios, at 1024 by 1024 with 20
denoising steps and classifier-free guidance 4.5. The optional cell S5b
checks that channel identity also holds at 10 steps. Text enters through
cross-attention, so the block sequence holds only the 4096 image tokens.
Each step runs one batch-2 forward with rows [negative, positive], and we
measure the conditional row.

Everything is measured at one block l\* and one denoising step t\*. A
five-scenario pilot over all blocks picks both with rules that are fixed in
the config before the pilot runs. The search band is blocks 13 to 20, with
ties going to block 13. As a pipeline check, the highest-norm token at the
probe block has to sit in the bottom-right corner of the image in every
pilot.

At the chosen block the image hidden state X (4096 tokens by 1152 channels)
is reduced to five per-channel statistics: signed mean, mean absolute value,
max absolute value, 99th percentile of the absolute value, and fraction of
positive entries. The pilot compares the rankings by |signed mean| and by
mean absolute value and fixes which one is primary before the main run.

Outputs are the scenario-averaged token by channel landscape (D1), the
recurrence f_k of per-scenario top-k channels for k = 1, 5 and 10 (D2), a
table of candidate channels (D3) and robustness checks across prompts,
seeds and denoising steps (D4).

Capture can be resumed after a disconnect, and every scenario is audited
before analysis. Each output file carries a hash of the config.

## How to run

1. The PixArt-Sigma weights are public, so no gated access is needed. A
   Hugging Face token is used only if one is set, either as the Colab secret
   `HF_TOKEN` or in the environment.
2. Pick a GPU runtime. An A100 runs everything quickly, but the 0.6B
   transformer also fits smaller GPUs in fp16. T5-XXL is loaded only once,
   to fill the embedding cache.
3. Run the cells from top to bottom. Each stage checks for the one before it.
   The smoke test must pass before the pilot, Phase B needs
   `pilot_decision.json`, and the analysis cells need the audit.
4. After a disconnect, reconnect and run all cells again. Finished work is
   found and skipped.
5. To only redraw the figures of a finished run, still use a GPU runtime,
   because the environment cell needs one. Run S0 to S2, which include LIB A
   and LIB F, and the library cells LIB B, C, D and E. Then define
   `PILOT_SIDS = [f"pilot__{scenario_id(p, s)}" for p, s in CFG.pilot_pairs]`
   (S4 normally sets it) and run S5, S8 and S9 to S13. Skip S3, S4 and S5b,
   which load or run the model. The S9c capture cell only needs the model if
   no matching capture exists. S5 rewrites `pilot_decision.json` from the
   stored pilot data with the same decision, which removes the S5b result
   from that file.

On an A100 the pilot takes a few minutes and the main run roughly 20 to 40
minutes. The run uses about 1.5 GiB on Drive. Figures are saved as a vector
PDF and a 600 dpi PNG, each with a draft caption in a `.caption.txt` file
next to it.

## S0. Environment

Installs the libraries with version bounds (Colab's preinstalled torch is left
alone), mounts Drive, checks free space and picks the precision mode from the
GPU. Ampere or newer GPUs run in bf16 and older ones in fp16, and the mode is
recorded in every output file. PixArt-Sigma is not gated, so no Hugging Face
token is needed.

In [ ]:
# =====================================================================
# S0. Environment setup
# =====================================================================
import importlib, importlib.util, subprocess, sys, os

def _pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"[setup][warn] pip install {' '.join(pkgs)} failed:\n{r.stderr[-800:]}")

# Leave torch alone. Colab's preinstalled build matches its GPUs.
_pip("diffusers>=0.32.0,<0.40", "transformers>=4.44.0", "accelerate>=0.33.0",
     "safetensors", "sentencepiece", "protobuf", "huggingface_hub>=0.24.0",
     "pyarrow", "pandas")

import gc, json, math, time, glob, re, shutil, hashlib, getpass, platform, datetime, csv
import numpy as np
import torch

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# ---------------------------------------------------------------- Drive
BASE = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = "/content/drive/MyDrive/pixart_ma_channels"
    except Exception as e:
        print(f"[setup][warn] Drive mount failed ({e}); falling back to local disk "
              f"-- outputs will NOT persist across sessions!")
if BASE is None:
    BASE = os.path.abspath("./pixart_ma_channels")

PATHS = {
    "base":    BASE,
    "meta":    os.path.join(BASE, "meta"),
    "embeds":  os.path.join(BASE, "prompt_embeds"),
    "pilot":   os.path.join(BASE, "pilot"),
    "main":    os.path.join(BASE, "main"),
    "smoke":   os.path.join(BASE, "smoke"),
    "figures": os.path.join(BASE, "figures"),
    "analysis": os.path.join(BASE, "analysis"),
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)
RUN_LOG_PATH   = os.path.join(PATHS["meta"], "run_log.csv")
ERROR_LOG_PATH = os.path.join(PATHS["meta"], "errors.log")
print(f"[setup] output root: {BASE}")

# ------------------------------------------------------- Drive preflight
try:
    du = shutil.disk_usage(BASE)
    free_gib = du.free / 2**30
    print(f"[setup] free space at output root: {free_gib:.1f} GiB "
          f"(experiment needs ~2-3 GiB)")
    if free_gib < 4:
        print("[setup][WARN] <4 GiB free -- clear Drive space before the main run.")
except Exception as e:
    print(f"[setup][warn] could not check disk usage: {e}")

# ---------------------------------------------------------- GPU detect
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU visible. In Colab: Runtime, Change runtime type, GPU, "
        "then restart and re-run this cell.")
GPU_PROPS  = torch.cuda.get_device_properties(0)
GPU_NAME   = GPU_PROPS.name
TOTAL_GIB  = GPU_PROPS.total_memory / 2**30
CC         = (GPU_PROPS.major, GPU_PROPS.minor)
DTYPE      = torch.bfloat16 if CC >= (8, 0) else torch.float16
DEVICE     = "cuda"

# The 0.6B PixArt transformer needs no quantization on any recent GPU.
PRECISION_MODE = "bf16_full" if DTYPE == torch.bfloat16 else "fp16_full"

print(f"[setup] GPU: {GPU_NAME} | {TOTAL_GIB:.1f} GiB | compute capability {CC[0]}.{CC[1]}")
print(f"[setup] precision mode: {PRECISION_MODE} | activation dtype at hooks: fp32 "
      f"(model dtype {DTYPE})")
if PRECISION_MODE != "bf16_full":
    print("[setup][note] pre-Ampere GPU: running in fp16. Channel identities "
          "are expected to be stable; magnitudes may shift slightly relative "
          "to bf16. Recorded as precision_mode in every artifact.")

# ---------------------------------------------------- Hugging Face access
# PixArt-Sigma is public, so a token is only used if one is present.
HF_TOKEN = None
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        if HF_TOKEN:
            print("[setup] HF token loaded from Colab secret 'HF_TOKEN'.")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or None

MODEL_ID = "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS"
from huggingface_hub import model_info
try:
    _ = model_info(MODEL_ID, token=HF_TOKEN)
    print(f"[setup] Hugging Face access to {MODEL_ID}: OK")
except Exception as e:
    raise RuntimeError(
        f"Cannot access {MODEL_ID} (a public model; check the network and "
        f"Hugging Face status, then re-run this cell). "
        f"Underlying error: {e}") from e

# ------------------------------------------------- library version record
def _ver(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return "n/a"
LIB_VERSIONS = {k: _ver(k) for k in
                ("torch", "diffusers", "transformers", "accelerate",
                 "numpy", "huggingface_hub")}
LIB_VERSIONS["python"] = platform.python_version()
print("[setup] versions:", json.dumps(LIB_VERSIONS))


## S1. Experiment config

All experiment parameters live in one frozen dataclass, and its hash is
stored in every output file. The rules for picking the analysis block and
step are part of this config and are fixed before the pilot. They are the
search band (blocks 13 to 20), the tie priority (block 13), the analysis step
(half way through sampling, or three quarters as the fallback), the
statistic-agreement thresholds and the position anchor on the bottom-right
high-norm token. The pilot only supplies the data these rules are applied to.

In [ ]:
# =====================================================================
# S1. Experiment config
#
# The config hash is stored in every output file.
# =====================================================================
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class ExperimentConfig:
    # -- model and sampling -----------------------------------------------
    model_id: str = "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS"
    height: int = 1024                    # -> 64x64 = 4096 image tokens (VAE /8, patchify 2x2)
    width: int = 1024
    num_steps: int = 20                   # standard PixArt-Sigma setting (DPM-Solver),
                                          # S5b compares against a 10-step setting
    guidance_scale: float = 4.5           # standard classifier-free guidance scale
    max_seq_len: int = 300                # T5 text token budget of PixArt-Sigma
    # -- guidance / batching ----------------------------------------------
    guidance_mode: str = "cfg_batch2"     # one batch-2 forward per step, rows [negative, positive]
    cond_row: int = 1                     # conditional row of the CFG batch, checked in S3c
    n_txt_in_seq: int = 0                 # text enters via cross-attention, never the block sequence
    # -- scenario grid -----------------------------------------------------
    seeds: tuple = (0, 42, 100, 1234, 2024)
    n_prompts: int = 20
    prompt_source_seed: int = 1234        # RNG seed for DiffusionDB stratified sampling
    # -- architecture expectations (PixArt-Sigma-XL-2) ---------------------
    d_model: int = 1152
    n_blocks_expected: int = 28
    # -- analysis block and step selection (rules fixed before the pilot) ---
    band: tuple = (13, 20)                # image-stream high-norm band, the search range for l*
    block_tie_priority: tuple = (13,)     # band onset
    block_tie_window_log10: float = 0.1   # blocks within 0.1 log10 of best count as ties
    t_star_default: int = 10              # round(0.5 * num_steps), half way through the 20 steps
    t_star_fallback: int = 15             # round(0.75 * num_steps)
    tstar_mag_frac_min: float = 0.6       # top-1 magnitude at t* >= 60% of the final step's
    tstar_stab_jaccard_min: float = 0.6   # top-10 overlap, t* vs final step
    # -- statistic-agreement gate (P4, checked on the pilot before Phase B) ---
    primary_stat_default: str = "abs_signed_mean"   # |mean| (Turri et al., Eq. 1)
    stat_gate_jaccard_min: float = 0.8
    sign_consistency_min: float = 0.9
    # -- position anchor check (P5) ---------------------------------
    # No massive-activation channel has been published for PixArt-Sigma, so
    # the check uses the bottom-right high-norm token instead. In every pilot
    # scenario the max-norm token at the probe block has to fall inside the
    # bottom-right anchor_region x anchor_region corner of the 64x64 grid.
    anchor_probe_block: int = 13
    anchor_grid: int = 64
    anchor_region: int = 8
    allow_missing_anchor: bool = False    # only set True after debugging by hand
    dg_anchor_channels: tuple = ()        # none exist for PixArt-Sigma, kept so the D3 column is defined
    # -- bridge to a 10-step setting (S5b) ----------------------------------
    bridge_steps: int = 10
    # -- logging ------------------------------------------------------------
    secondary_blocks: tuple = (3, 8, 13, 16, 20, 24)
    # 3 early control, 8 pre-onset control, 13 band onset, 16 band middle,
    # 20 band end, 24 post-band control
    k_list: tuple = (1, 5, 10)
    top_save: int = 20
    save_full_tensor: bool = True         # fp16 [4096,1152] at (l*, t*) per scenario (~9 MB)
    # -- pilot & smoke ------------------------------------------------------
    pilot_pairs: tuple = (("p00", 42), ("p01", 0), ("p02", 1234),
                          ("p03", 100), ("p04", 2024))
    smoke_steps: int = 4
    recurring_set_threshold: float = 0.9  # G_k = {d : f_k(d) >= this}

CFG = ExperimentConfig()
N_IMG = (CFG.height // 8 // 2) * (CFG.width // 8 // 2)   # VAE /8, patchify 2x2
assert N_IMG == 4096, f"unexpected token count {N_IMG}; resolution changed?"

CONFIG_HASH = hashlib.sha256(
    json.dumps(asdict(CFG), sort_keys=True, default=str).encode()).hexdigest()[:12]

print(f"[config] {CFG.n_prompts} prompts x {len(CFG.seeds)} seeds = "
      f"{CFG.n_prompts * len(CFG.seeds)} scenarios | {CFG.num_steps} steps @ "
      f"{CFG.height}x{CFG.width} ({N_IMG} image tokens, D={CFG.d_model})")
print(f"[config] hash: {CONFIG_HASH}")
for k, v in asdict(CFG).items():
    print(f"    {k:26s} = {v}")


### LIB A. Core utilities

Small helpers used throughout. They cover Jaccard, top-k and rank functions,
the primary score and sign consistency, atomic writes (write a temp file,
then rename), scenario paths, the completeness check that lets a run resume,
and the run-log columns.


In [ ]:
# =====================================================================
# LIB A. Core utilities
#
# numpy and the standard library only.
# =====================================================================
import os, json, csv, hashlib, datetime
import numpy as np

STAT_NAMES = ("signed_mean", "mean_abs", "max_abs", "p99_abs", "frac_pos")

# ------------------------------------------------------------ set math
def jaccard(a, b):
    """Jaccard overlap of two iterables of channel ids."""
    A, B = set(int(x) for x in a), set(int(x) for x in b)
    if not A and not B:
        return 1.0
    return len(A & B) / len(A | B)

def topk_ids(score_vec, k):
    """Indices of the k largest scores, largest first. Ties go to the lower id."""
    s = np.asarray(score_vec, dtype=np.float64)
    # stable sort, so tied ids stay in ascending order
    order = np.argsort(-s, kind="stable")
    return order[:k].astype(np.int64)

def rank_of(score_vec, channel):
    """Competition rank (1 = largest) of `channel` within score_vec."""
    s = np.asarray(score_vec, dtype=np.float64)
    return 1 + int((s > s[int(channel)]).sum())

def primary_score(stats_at_t, primary_stat):
    """Primary score [D] (float64) at one block and step.

    stats_at_t maps a stat name to its [D] vector. primary_stat is
    'abs_signed_mean' or 'mean_abs'."""
    if primary_stat == "abs_signed_mean":
        return np.abs(np.asarray(stats_at_t["signed_mean"], dtype=np.float64))
    if primary_stat == "mean_abs":
        return np.asarray(stats_at_t["mean_abs"], dtype=np.float64)
    raise ValueError(f"unknown primary_stat {primary_stat!r}")

def sign_consistency(frac_pos):
    """1.0 if the channel keeps one sign on every token, 0.5 if fully mixed."""
    f = np.asarray(frac_pos, dtype=np.float64)
    return np.maximum(f, 1.0 - f)

# ------------------------------------------------------------ atomic IO
def save_json_atomic(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)

def save_npz_atomic(path, **arrays):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.savez_compressed(f, **arrays)
    os.replace(tmp, path)

def save_npy_atomic(path, arr):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)

def save_png_atomic(pil_image, path):
    tmp = path + ".tmp.png"
    pil_image.save(tmp, format="PNG")
    os.replace(tmp, path)

def meta_to_npz_field(meta_dict):
    return np.array(json.dumps(meta_dict, default=str))

def meta_from_npz(z):
    return json.loads(str(z["meta_json"].item()))

# --------------------------------------------------- scenario bookkeeping
def scenario_id(pid, seed):
    return f"{pid}_s{seed}"

def scenario_paths(out_dir, sid, save_tensor, lstar=None, tstar=None):
    p = {"npz": os.path.join(out_dir, f"{sid}.npz"),
         "png": os.path.join(out_dir, f"{sid}.png"),
         "tensor": None}
    if save_tensor:
        assert lstar is not None and tstar is not None, \
            "tensor capture requested but (l*, t*) not set"
        p["tensor"] = os.path.join(out_dir, f"{sid}__Xfull_l{lstar:02d}_t{tstar:02d}.npy")
    return p

def is_scenario_complete(paths, need_tensor, expect=None):
    """True if the npz and png (and the tensor, when needed) exist and the npz
    opens with the required keys. If `expect` holds meta fields such as lstar,
    tstar or primary_stat, the stored meta has to match them too, so a resume
    doesn't keep results made under an older pilot decision."""
    if not (os.path.exists(paths["npz"]) and os.path.exists(paths["png"])):
        return False
    if need_tensor and not (paths["tensor"] and os.path.exists(paths["tensor"])):
        return False
    try:
        with np.load(paths["npz"], allow_pickle=False) as z:
            if not (("meta_json" in z.files) and ("sigmas" in z.files)):
                return False
            if expect:
                m = meta_from_npz(z)
                for k, v in expect.items():
                    if m.get(k) != v:
                        return False
            return True
    except Exception:
        return False

# ---------------------------------------------------------- run log CSV
RUN_LOG_FIELDS = [
    "timestamp", "tag", "sid", "prompt_id", "seed", "status", "wall_s",
    "peak_vram_gib", "fires_ok", "nan_flag", "hook_errors", "tensor_captured",
    "top1_channel", "top1_score", "top1_over_median", "top1_sign_consistency",
    "jacc_primary_vs_meanabs_top10", "precision_mode",
]

def append_run_log(csv_path, row_dict):
    new = not os.path.exists(csv_path)
    row = {k: row_dict.get(k, "") for k in RUN_LOG_FIELDS}
    row["timestamp"] = row.get("timestamp") or datetime.datetime.now().isoformat(timespec="seconds")
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RUN_LOG_FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)

def append_error_log(path, sid, err_text):
    with open(path, "a") as f:
        f.write(f"[{datetime.datetime.now().isoformat(timespec='seconds')}] {sid}: {err_text}\n\n")

def sha256_of_text(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]

print("[lib-core] loaded:", ", ".join(
    ["jaccard", "topk_ids", "rank_of", "primary_score", "sign_consistency",
     "atomic IO", "scenario paths", "run log"]))


### LIB F. Figure style

Every figure gets a short title placed clear of the channel labels and legend
(`fig_title`), and its caption draft goes to a `.caption.txt` file next to it
for use in LaTeX. Categorical colors use the colorblind-safe Okabe-Ito
palette. Magnitude heatmaps use sequential colormaps with a log norm, and the
diverging colormap is only used for signed data. Each figure is saved as a
vector PDF and a 600 dpi PNG, with TrueType (Type 42) fonts sized for a single
column (3.35 in) or the full page width (7 in).

In [ ]:
# =====================================================================
# LIB F. Figure style
#
# Each figure gets a short title placed clear of channel labels and
# legends (fig_title), and a caption draft in <stem>.caption.txt for LaTeX.
# Categorical colors follow the Okabe-Ito palette. Magnitude heatmaps use
# sequential colormaps with a log norm, and signed data uses a dark-centered
# diverging colormap. Each figure is saved as a vector PDF plus a 600 dpi
# PNG, with TrueType (Type 42) fonts sized for a column or the full page.
#
# For color coding, each channel is put in a category by comparing a
# per-channel magnitude statistic with the median channel. This follows the
# massive-activations habit of calling a value an outlier when it is orders
# of magnitude above the typical one.
#   massive   statistic >= MASSIVE_MULT x median   (default 100x)
#   high      statistic >= HIGH_MULT x median      (default 10x)
#   typical   everything else
# Captions that use these colors state the multipliers.
# =====================================================================
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm

FIG_COL, FIG_PAGE = 3.35, 7.00      # inches, single column and full width
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00",
             "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
_BASE_PT = 8.0

def use_paper_style():
    mpl.rcParams.update({
        "figure.constrained_layout.use": True,
        "figure.dpi": 110,
        "pdf.fonttype": 42, "ps.fonttype": 42,
        "font.size": _BASE_PT,
        "axes.titlesize": _BASE_PT, "axes.labelsize": _BASE_PT,
        "xtick.labelsize": _BASE_PT - 1, "ytick.labelsize": _BASE_PT - 1,
        "legend.fontsize": _BASE_PT - 1, "legend.frameon": False,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "xtick.major.size": 2.5, "ytick.major.size": 2.5,
        "lines.linewidth": 1.0,
        "axes.prop_cycle": mpl.cycler(color=OKABE_ITO),
    })

use_paper_style()

MODEL_LABEL = {
    "black-forest-labs/FLUX.1-dev": "FLUX.1-dev",
    "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS": "PixArt-Sigma",
    "stabilityai/stable-diffusion-3-medium-diffusers": "SD3-Medium",
}.get(CFG.model_id, CFG.model_id)

BLOCK_AXIS_LABEL = {
    "FLUX.1-dev": "block index (0 to 18 dual, 19 to 56 single)",
}.get(MODEL_LABEL, "block index")

PRIMARY_TEX = {"mean_abs": r"mean $|x|$",
               "abs_signed_mean": r"$|\mathrm{mean}\,x|$"}
PRIMARY_WORDS = {"mean_abs": "the per-channel mean absolute activation",
                 "abs_signed_mean": "the absolute per-channel signed mean"}

# ---------------- channel categorization (high / massive color coding)
HIGH_MULT, MASSIVE_MULT = 10.0, 100.0
# profile figures on a white background use warm accent colors
CAT_COLORS = {0: "#6A51A3", 1: "#E69F00", 2: "#D55E00"}
# 3D bar landscape uses a blue floor and purples for the outlier tiers
BAR3D_COLORS = {0: "#6BAED6", 1: "#9E8FD0", 2: "#5C2D91"}
CAT_NAMES = {0: "typical",
             1: f"high (>= {HIGH_MULT:g}x median)",
             2: f"massive (>= {MASSIVE_MULT:g}x median)"}
CAT_CAPTION = (f"Channels are categorized relative to the median channel of "
               f"the same statistic: massive is at least {MASSIVE_MULT:g} "
               f"times the median, high is at least {HIGH_MULT:g} times the "
               f"median.")

def classify_channels(vec, high_mult=None, massive_mult=None):
    """[D] magnitude statistic -> (categories [D] int8, median).

    2 is massive, 1 high, 0 typical. Thresholds are multiples of the median
    channel and default to HIGH_MULT and MASSIVE_MULT."""
    v = np.asarray(vec, np.float64)
    med = max(float(np.median(v)), 1e-12)
    hm = HIGH_MULT if high_mult is None else float(high_mult)
    mm = MASSIVE_MULT if massive_mult is None else float(massive_mult)
    cat = np.zeros(v.size, np.int8)
    cat[v >= hm * med] = 1
    cat[v >= mm * med] = 2
    return cat, med

def _make_signed_cmap():
    """Black-centered diverging colormap. The positive half is magma, as in
    the magnitude heatmaps, and the negative half is a matching ramp from
    black through blue and cyan to pale cyan, so all landscapes look alike."""
    from matplotlib.colors import ListedColormap, LinearSegmentedColormap
    pos = plt.get_cmap("magma")(np.linspace(0.0, 1.0, 256))
    neg_ramp = LinearSegmentedColormap.from_list(
        "negside", ["#000004", "#1D1147", "#1F4287", "#2D7DD2",
                    "#40C9E0", "#B5F2F5"])
    neg = neg_ramp(np.linspace(0.0, 1.0, 256))[::-1]
    return ListedColormap(np.vstack([neg, pos]), name="signed_magma")

SIGNED_CMAP = _make_signed_cmap()

def fig_stem(name):
    return os.path.join(PATHS["figures"], name)

def save_fig(fig, stem, caption=None, show=True, tight=True):
    """Write <stem>.pdf, <stem>.png and, with a caption, <stem>.caption.txt
    atomically.

    Use tight=False for 3D axes. The tight bounding box under-measures 3D
    axis labels and crops them, and their placement drifts a little with
    the raster dpi, so 3D figures skip the tight box and set their margins
    with ax.set_position instead."""
    for ext, dpi in (("pdf", None), ("png", 600)):
        tmp = f"{stem}.tmp.{ext}"
        fig.savefig(tmp, bbox_inches="tight" if tight else None, dpi=dpi)
        os.replace(tmp, f"{stem}.{ext}")
    if caption:
        tmp = f"{stem}.caption.txt.tmp"
        with open(tmp, "w") as f:
            f.write(caption.strip() + "\n")
        os.replace(tmp, f"{stem}.caption.txt")
    print(f"[fig] saved {os.path.basename(stem)}"
          + (" (.pdf/.png/.caption.txt)" if caption else " (.pdf/.png)"))
    if show:
        plt.show()

# Color floor percentile for the magnitude heatmaps. At 1.0 the whole line
# texture of the landscape stays visible. Raise it toward 90 to black out
# the weaker lines and show only the strongest channels.
HEAT_FLOOR_PCTL = 1.0

def _cluster_sides(xs_sorted, span, sep_frac=0.04):
    """Pick a label side for each sorted point. Points closer than sep_frac
    of the axis span form a cluster. A lone point is labeled centered
    above, a pair gets left and right, and a larger cluster gets left, then
    center for the middle points, then right."""
    sep = sep_frac * float(span)
    sides, cluster = [], [0]

    def _flush(c):
        if len(c) == 1:
            sides.append("center")
        elif len(c) == 2:
            sides.extend(["left", "right"])
        else:
            sides.append("left")
            sides.extend(["center"] * (len(c) - 2))
            sides.append("right")

    for i in range(1, len(xs_sorted)):
        if xs_sorted[i] - xs_sorted[i - 1] <= sep:
            cluster.append(i)
        else:
            _flush(cluster)
            cluster = [i]
    _flush(cluster)
    return sides

def annotate_points(ax, xs, ys, labels):
    """Channel-id labels at their points. A lone channel is labeled right
    above its point. Two neighboring channels are labeled to the left and
    right of their points, so the pair stays readable without extra marks."""
    xs = np.asarray(xs, dtype=np.float64)
    ys = np.asarray(ys, dtype=np.float64)
    order = np.argsort(xs)
    xmin, xmax = ax.get_xlim()
    sides = _cluster_sides([float(xs[k]) for k in order], xmax - xmin)
    off = {"left": (-5, 1), "center": (0, 5), "right": (5, 1)}
    ha = {"left": "right", "center": "center", "right": "left"}
    for k, side in zip(order, sides):
        ax.annotate(str(labels[k]), (xs[k], ys[k]),
                    xytext=off[side], textcoords="offset points",
                    ha=ha[side], va="bottom", fontsize=_BASE_PT - 1.5)

def label_channels_top(ax, positions, labels, color="0.15"):
    """Channel-id labels in one row above the axes. Labels that would overlap
    are pushed sideways just enough to clear each other, using widths
    estimated from the font size and axes width. A thin leader line ties
    each label to its exact channel position."""
    if len(positions) == 0:
        return
    xs = np.asarray(positions, dtype=np.float64)
    order = np.argsort(xs)
    xs = xs[order]
    labs = [str(int(np.asarray(labels)[k])) for k in order]
    xmin, xmax = sorted(ax.get_xlim())
    fs = _BASE_PT - 3
    ax_pt = max(ax.figure.get_figwidth() * ax.get_position().width * 72.0,
                1.0)
    upp = (xmax - xmin) / ax_pt                    # data units per point
    wid = [(0.62 * fs * len(t) + 3.0) * upp for t in labs]
    pos = list(xs)
    for _ in range(200):                           # 1D dodge until no labels overlap
        moved = False
        for i in range(1, len(pos)):
            need = 0.5 * (wid[i - 1] + wid[i])
            if pos[i] - pos[i - 1] < need:
                mid = 0.5 * (pos[i] + pos[i - 1])
                pos[i - 1] = mid - 0.5 * need
                pos[i] = mid + 0.5 * need
                moved = True
        for i in range(len(pos)):                  # stay inside the axes
            pos[i] = min(max(pos[i], xmin + 0.5 * wid[i]),
                         xmax - 0.5 * wid[i])
        if not moved:
            break
    for x0, xl, t in zip(xs, pos, labs):
        ax.annotate(t, xy=(x0, 1.0), xycoords=("data", "axes fraction"),
                    xytext=((xl - x0) / upp, 8.0),
                    textcoords="offset points",
                    ha="center", va="bottom", fontsize=fs, color=color,
                    annotation_clip=False,
                    arrowprops=dict(arrowstyle="-", lw=0.5, color=color,
                                    shrinkA=1.0, shrinkB=0.0))

def legend_above(ax, ncol=2, title=None):
    """Legend in a strip above the axes, never inside the data area."""
    if ax.get_legend_handles_labels()[0]:
        ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.02), ncol=ncol,
                  borderaxespad=0.0, frameon=False, columnspacing=1.0,
                  handletextpad=0.4, title=title,
                  title_fontsize=_BASE_PT - 1)

def fig_title(ax, text, pad=6):
    """Figure title above the axes. The strip above the axes may already
    hold channel-id labels or a legend, so we draw once, measure how far
    everything there reaches and put the title just above it. `pad` is the
    minimum gap in points for a bare axes."""
    fig = ax.figure
    try:
        fig.canvas.draw()                      # draw so the extents are known
        ren = fig.canvas.get_renderer()
        ax_top = ax.get_window_extent(ren).y1
        top = ax_top
        arts = list(ax.texts)
        if ax.get_legend() is not None:
            arts.append(ax.get_legend())
        for a in arts:
            try:
                top = max(top, a.get_window_extent(ren).y1)
            except Exception:
                pass
        pad = max(pad, (top - ax_top) * 72.0 / fig.dpi + 6.0)
    except Exception:
        pass
    ax.set_title(text, fontsize=_BASE_PT + 0.5, pad=pad, loc="center")

def category_profile(ax, vec, cats, top_ids=None, label_top=True):
    """Log-scale per-channel profile with category color coding."""
    D = len(vec)
    ax.semilogy(np.arange(D), vec, lw=0.4, color="0.75", zorder=1)
    for c in (1, 2):
        idx = np.where(cats == c)[0]
        if len(idx):
            ax.scatter(idx, np.asarray(vec)[idx], s=10 if c == 2 else 5,
                       color=CAT_COLORS[c], zorder=3, label=CAT_NAMES[c],
                       linewidths=0)
    ax.set_xlim(0, D)
    if top_ids is not None and label_top:
        annotate_points(ax, list(top_ids),
                        [float(np.asarray(vec)[c]) for c in top_ids],
                        [int(c) for c in top_ids])

print(f"[lib-fig] paper style active | model label: {MODEL_LABEL} | "
      f"signed cmap: {SIGNED_CMAP.name}")


## S2. Prompt manifest and scenario grid
Prompts p00 to p02 are three fixed bridge prompts, used as written. Prompts
p03 to p19 come from the DiffusionDB metadata table (text and NSFW scores
only, no images). We keep ASCII prompts of 5 to 25 words with both NSFW
scores below 0.2, drop anything on a small blocklist or with more than four
comma-separated parts, and remove duplicates after normalizing case and
punctuation. The survivors are sampled round-robin over eight content
buckets so the set isn't dominated by one kind of scene.

If the download fails, a curated fallback list is used instead. In both cases
the list is written once to `prompts.json` with a sha-256 digest, and later
sessions reload that file instead of sampling again.

If the FLUX.1-dev run's `prompts.json` is already on Drive, it is copied
first, so both models get exactly the same prompts.

In [ ]:
# =====================================================================
# S2. Prompt manifest and scenario grid
# =====================================================================
# p00 to p02 are the three fixed bridge prompts. p03 to p19 are filtered
# from poloclub/diffusiondb metadata.parquet and sampled over 8 content
# buckets, with a curated fallback list if that fails. The list is written
# to prompts.json on the first build and reloaded, not resampled, after that.
import random

TABLE6_PROMPTS = [
    "a cat holding hello world sign",
    "an astronaut in the moon holding the USA flag",
    "a protestor in a city square holding a 'change' sign",
]

FALLBACK_PROMPTS = [
    "a close-up portrait of an elderly fisherman with a weathered face, dramatic lighting",
    "a red fox standing in a snowy birch forest at dawn",
    "a wide mountain valley with a winding river under storm clouds",
    "a cozy cluttered artist studio with paint tubes and canvases by a window",
    "a vintage brass pocket watch on a wooden desk, macro photograph",
    "a rusty pickup truck parked beside a desert gas station at sunset",
    "a neon sign that says 'open all night' above a diner entrance",
    "an isometric illustration of a tiny floating island village, pastel colors",
    "a chef plating dessert in a busy restaurant kitchen",
    "a lighthouse on a cliff during a thunderstorm, long exposure",
    "a black cat sleeping on a stack of old books",
    "an astronaut greenhouse on mars with rows of green plants",
    "a watercolor painting of a rainy tokyo street at night",
    "a medieval knight's helmet on a stone pedestal in a museum",
    "a hot air balloon festival over lavender fields",
    "a robot barista serving coffee in a futuristic cafe",
    "an oil painting of a stormy sea with a small sailboat",
]

BUCKET_KEYWORDS = {
    "portrait":  ("portrait", "face", " man ", " woman ", " girl ", " boy ", "person"),
    "animal":    ("cat", "dog", "fox", "bird", "horse", "wolf", "tiger", "lion", "deer", "animal"),
    "landscape": ("mountain", "forest", "valley", "landscape", "river", "desert",
                  "beach", "sunset", "field", "waterfall"),
    "indoor":    ("room", "kitchen", "interior", "office", "studio", "library", "bedroom"),
    "object":    ("watch", "bottle", "chair", "table", "lamp", "book", "sword",
                  "cup", "clock", "vase"),
    "vehicle":   ("car", "truck", "ship", "train", "plane", "motorcycle", "boat", "bus"),
    "text_in_image": ("sign", " text", "poster", "billboard", "label", "graffiti"),
    "stylized":  ("watercolor", "oil painting", "isometric", "pixel art",
                  "illustration", "anime", "sketch", "low poly"),
}

_BLOCKLIST_RE = re.compile(
    r"\b(nsfw|nude|naked|topless|porn|sexy|erotic|hentai|lingerie|gore|blood|"
    r"beheading|corpse|mutilat\w*|loli|shota)\b", re.IGNORECASE)

def _normalize_prompt(p):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", str(p).lower())).strip()

def _passes_filters(p, p_nsfw, i_nsfw):
    if not isinstance(p, str):
        return False
    p = p.strip()
    if not (p and p.isascii()):
        return False
    n_words = len(p.split())
    if not (5 <= n_words <= 25):
        return False
    try:
        if float(p_nsfw) >= 0.2 or float(i_nsfw) >= 0.2:
            return False
    except (TypeError, ValueError):
        return False
    if _BLOCKLIST_RE.search(p):
        return False
    if p.count(",") + 1 > 4:            # at most 4 comma-separated segments
        return False
    return True

def _bucket_of(p):
    q = f" {p.lower()} "
    for b, kws in BUCKET_KEYWORDS.items():
        if any(kw in q for kw in kws):
            return b
    return "other"

def select_diffusiondb_prompts(n_needed, rng_seed, exclude_norms):
    """Download the DiffusionDB metadata (prompt text and NSFW scores, no
    images), filter and dedupe it, then pick n_needed prompts round-robin
    over the content buckets."""
    from huggingface_hub import hf_hub_download
    import pyarrow.parquet as pq
    print("[prompts] downloading DiffusionDB metadata.parquet (~0.5 GB, one time) ...")
    fp = hf_hub_download("poloclub/diffusiondb", "metadata.parquet", repo_type="dataset")
    tbl = pq.read_table(fp, columns=["prompt", "prompt_nsfw", "image_nsfw"])
    df = tbl.to_pandas()
    print(f"[prompts] {len(df):,} raw rows")
    df = df[df["prompt"].map(lambda p: _passes_filters(p, 0.0, 0.0))]  # cheap text filters first
    mask = [( _passes_filters(p, pn, im)) for p, pn, im in
            zip(df["prompt"], df["prompt_nsfw"], df["image_nsfw"])]
    df = df[np.asarray(mask, dtype=bool)]
    print(f"[prompts] {len(df):,} rows after filters")
    seen, rows = set(exclude_norms), []
    for p in df["prompt"]:
        k = _normalize_prompt(p)
        if k and k not in seen:
            seen.add(k)
            rows.append(p.strip())
    buckets = {b: [] for b in list(BUCKET_KEYWORDS) + ["other"]}
    for p in rows:
        buckets[_bucket_of(p)].append(p)
    rng = random.Random(rng_seed)
    for b in buckets:
        rng.shuffle(buckets[b])
    order = list(BUCKET_KEYWORDS) + ["other"]
    picked, provenance = [], []
    while len(picked) < n_needed:
        progressed = False
        for b in order:
            if buckets[b] and len(picked) < n_needed:
                p = buckets[b].pop()
                picked.append(p)
                provenance.append(b)
                progressed = True
        if not progressed:
            raise RuntimeError("DiffusionDB pool exhausted before filling the manifest")
    return picked, provenance

def build_or_load_prompts(cfg, meta_dir):
    """Return ({pid: text}, manifest dict). The manifest is fixed after the
    first build."""
    path = os.path.join(meta_dir, "prompts.json")
    if os.path.exists(path):
        with open(path) as f:
            man = json.load(f)
        prompts = {e["pid"]: e["text"] for e in man["prompts"]}
        assert len(prompts) == cfg.n_prompts, \
            f"frozen prompts.json has {len(prompts)} prompts, config expects {cfg.n_prompts}"
        got = sha256_of_text("\n".join(prompts[f"p{i:02d}"] for i in range(cfg.n_prompts)))
        assert got == man["sha256"], "prompts.json content/sha mismatch -- file corrupted?"
        print(f"[prompts] loaded FROZEN manifest ({man['source']}, sha {man['sha256']})")
        return prompts, man
    n_extra = cfg.n_prompts - len(TABLE6_PROMPTS)
    exclude = {_normalize_prompt(p) for p in TABLE6_PROMPTS}
    try:
        extra, prov = select_diffusiondb_prompts(n_extra, cfg.prompt_source_seed, exclude)
        source = "table6+diffusiondb"
    except Exception as e:
        print(f"[prompts][warn] DiffusionDB path failed ({type(e).__name__}: {e})")
        print("[prompts] using curated fallback list (source recorded in manifest).")
        extra, prov = FALLBACK_PROMPTS[:n_extra], ["curated"] * n_extra
        source = "table6+curated_fallback"
    texts = TABLE6_PROMPTS + extra
    assert len(texts) == cfg.n_prompts, \
        f"prompt build produced {len(texts)} prompts, config expects {cfg.n_prompts}"
    provs = ["table6_bridge"] * len(TABLE6_PROMPTS) + prov
    entries = [{"pid": f"p{i:02d}", "text": t, "bucket": b}
               for i, (t, b) in enumerate(zip(texts, provs))]
    man = {"source": source, "rng_seed": cfg.prompt_source_seed,
           "built": datetime.datetime.now().isoformat(timespec="seconds"),
           "sha256": sha256_of_text("\n".join(texts)), "prompts": entries}
    save_json_atomic(man, path)
    print(f"[prompts] built & FROZE manifest -> {path} (source={source})")
    return {e["pid"]: e["text"] for e in entries}, man

# ---- reuse the prompts.json already on Drive (paths below) so both models get the same prompts
_local_manifest = os.path.join(PATHS["meta"], "prompts.json")
if not os.path.exists(_local_manifest):
    for _cand in ("/content/drive/MyDrive/flux_ma_channels/meta/prompts.json",
                  os.path.abspath("./flux_ma_channels/meta/prompts.json")):
        if os.path.exists(_cand):
            shutil.copy(_cand, _local_manifest)
            print(f"[prompts] copied frozen manifest from {_cand}")
            print("[prompts] cross-model comparability requires the same "
                  "manifest sha in every model's run manifest")
            break

PROMPTS, PROMPT_MANIFEST = build_or_load_prompts(CFG, PATHS["meta"])

# ---------------- scenario grid, prompt-major (all seeds of p00, then p01, ...)
SCENARIOS = [(scenario_id(pid, s), pid, s, PROMPTS[pid])
             for pid in sorted(PROMPTS) for s in CFG.seeds]
import pandas as pd
scen_df = pd.DataFrame(SCENARIOS, columns=["sid", "prompt_id", "seed", "prompt"])
scen_df.to_csv(os.path.join(PATHS["meta"], "scenarios.csv"), index=False)
print(f"[prompts] {len(SCENARIOS)} scenarios written to scenarios.csv")
for pid in sorted(PROMPTS):
    print(f"    {pid}: {PROMPTS[pid]}")


## S3. Model loading and smoke test
Loading happens in two phases. S3a loads only the T5-XXL encoder and encodes
all 20 prompts once, together with the empty negative prompt and the
attention masks that classifier-free guidance needs. It saves the fp32
embeddings to Drive and unloads the encoder. Every seed of a prompt then gets
exactly the same conditioning, and a resumed session never loads T5 at all.
S3b loads the generation pipeline without the text encoder.

S3c is the smoke test. Under classifier-free guidance each forward pass has
a batch of 2, so it first checks that the row we treat as conditional
matches a guidance-free run at step 0, where the input latents are the same.
It then does a 4-step generation with hooks on all 28 blocks and checks the
block count, hook fire counts, finite statistics, and that the saved files
load back from disk. Nothing else should be run until it passes.

In [ ]:
# =====================================================================
# S3a. Prompt embedding cache
# =====================================================================
# Encode all 20 prompts once with T5-XXL, along with the empty negative
# prompt and the attention masks that classifier-free guidance needs. Save
# the fp32 embeddings to Drive and delete the text encoder. This frees VRAM,
# keeps the conditioning identical across seeds, and lets a resumed session
# skip T5.

def build_prompt_embed_cache(cfg, prompts, embed_dir, token, enc_dtype):
    embed_path = lambda pid: os.path.join(embed_dir, f"{pid}.npz")

    def _valid(pid):
        p = embed_path(pid)
        if not os.path.exists(p):
            return False
        try:
            with np.load(p, allow_pickle=False) as z:
                m = meta_from_npz(z)
                return (m["text_sha"] == sha256_of_text(prompts[pid])
                        and z["prompt_embeds"].shape[1] == cfg.max_seq_len
                        and "neg_embeds" in z.files)
        except Exception:
            return False

    missing = [pid for pid in sorted(prompts) if not _valid(pid)]
    if not missing:
        print(f"[embeds] all {len(prompts)} prompt embeddings cached & verified -- "
              f"the T5 encoder will NOT be loaded this session.")
        return
    print(f"[embeds] encoding {len(missing)} prompt(s): {missing}")
    from diffusers import PixArtSigmaPipeline
    pipe_text = PixArtSigmaPipeline.from_pretrained(
        cfg.model_id, transformer=None, vae=None,
        torch_dtype=enc_dtype, token=token)
    try:
        pipe_text.to("cuda")
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        pipe_text.enable_model_cpu_offload()
        print("[embeds][warn] T5 offloaded to CPU (small GPU); encoding is slower")
    with torch.inference_mode():
        for pid in missing:
            text = prompts[pid]
            pe, pm, ne, nm = pipe_text.encode_prompt(
                prompt=text, negative_prompt="", device="cuda",
                do_classifier_free_guidance=True,
                max_sequence_length=cfg.max_seq_len)
            assert pe.ndim == 3 and pe.shape[0] == 1 and pe.shape[1] == cfg.max_seq_len, \
                f"unexpected prompt_embeds shape {tuple(pe.shape)}"
            assert ne.shape == pe.shape, \
                f"negative embeds shape {tuple(ne.shape)} != {tuple(pe.shape)}"
            assert pm.shape[1] == cfg.max_seq_len and nm.shape[1] == cfg.max_seq_len
            save_npz_atomic(
                embed_path(pid),
                prompt_embeds=pe.float().cpu().numpy(),
                prompt_mask=pm.cpu().numpy().astype(np.int64),
                neg_embeds=ne.float().cpu().numpy(),
                neg_mask=nm.cpu().numpy().astype(np.int64),
                meta_json=meta_to_npz_field({
                    "pid": pid, "text": text, "text_sha": sha256_of_text(text),
                    "enc_dtype": str(enc_dtype), "max_seq_len": cfg.max_seq_len,
                    "model_id": cfg.model_id, "negative_prompt": ""}))
            print(f"    {pid}: encoded & saved  (T5 tokens={pe.shape[1]})")
    del pipe_text
    gc.collect(); torch.cuda.empty_cache()
    print("[embeds] text encoder unloaded; VRAM freed for generation.")

def load_embed_cache(prompts, embed_dir):
    """Return {pid: dict(pe, pm, ne, nm)} on CPU. The embeddings pe and ne
    are fp32 [1,300,4096] and the masks pm and nm are int64 [1,300]."""
    out = {}
    for pid in sorted(prompts):
        with np.load(os.path.join(embed_dir, f"{pid}.npz"), allow_pickle=False) as z:
            m = meta_from_npz(z)
            assert m["text_sha"] == sha256_of_text(prompts[pid]), \
                f"{pid}: cached embedding text mismatch -- delete {embed_dir} and re-run"
            out[pid] = {"pe": torch.from_numpy(z["prompt_embeds"].copy()),
                        "pm": torch.from_numpy(z["prompt_mask"].copy()),
                        "ne": torch.from_numpy(z["neg_embeds"].copy()),
                        "nm": torch.from_numpy(z["neg_mask"].copy())}
    return out

build_prompt_embed_cache(CFG, PROMPTS, PATHS["embeds"], HF_TOKEN, DTYPE)
EMBEDS = load_embed_cache(PROMPTS, PATHS["embeds"])
assert int(next(iter(EMBEDS.values()))["pe"].shape[1]) == CFG.max_seq_len
print(f"[embeds] loaded {len(EMBEDS)} embeddings into RAM "
      f"(T5 tokens={CFG.max_seq_len}; text is cross-attended, so block "
      f"sequences hold image tokens only)")


In [ ]:
# =====================================================================
# S3b. Generation pipeline
# =====================================================================
# The text encoder stays unloaded because the embeddings are cached. The
# 0.6B transformer fits in 16-bit on any recent GPU, so there is no
# quantized path. If loading runs out of memory we fall back to CPU offload,
# and the precision mode records that.

def load_generation_pipeline(cfg, dtype, token):
    from diffusers import PixArtSigmaPipeline
    pipe = PixArtSigmaPipeline.from_pretrained(
        cfg.model_id, text_encoder=None, tokenizer=None,
        torch_dtype=dtype, token=token)
    try:
        pipe.to("cuda")
        return pipe, PRECISION_MODE
    except torch.cuda.OutOfMemoryError:
        print("[pipe][warn] full-GPU load OOM'd -> enabling model CPU offload")
        gc.collect(); torch.cuda.empty_cache()
        pipe.enable_model_cpu_offload()
        return pipe, PRECISION_MODE + "_offload"

print(f"[pipe] loading {CFG.model_id} (mode={PRECISION_MODE}, dtype={DTYPE}) ...")
_t0 = time.time()
pipe, PRECISION_MODE = load_generation_pipeline(CFG, DTYPE, HF_TOKEN)
pipe.set_progress_bar_config(disable=True)
try:
    _pdev = next(pipe.transformer.parameters()).device
    _pdt = next(pipe.transformer.parameters()).dtype
except StopIteration:
    _pdev = _pdt = "n/a"
print(f"[pipe] ready in {time.time()-_t0:.0f}s | effective mode={PRECISION_MODE} | "
      f"transformer on {_pdev} ({_pdt}) | "
      f"VRAM allocated {torch.cuda.memory_allocated()/2**30:.1f} GiB")
print(f"[pipe] scheduler: {type(pipe.scheduler).__name__}")


### LIB B and LIB C. Measurement core
Forward hooks sit on the block outputs. The image stream is found by its
shape. PixArt blocks return one tensor that holds exactly the 4096 image
tokens, because text comes in through cross-attention. Under
classifier-free guidance the batch is 2 and we measure the conditional row
(index 1). Statistics are computed in fp32 on the GPU, so only five
D-dimensional vectors per block and step are copied to the CPU.

There is exactly one transformer forward per denoising step, so each
block's hook counter doubles as the step index. The counters are checked
when the recorder is finalized.

`run_scenario` is the one function used by the smoke test, the pilot and
the main run, so every phase measures the same way. It saves the image, then
the tensor, then the npz. The npz is written last and marks a scenario as
complete when resuming.

In [ ]:
# =====================================================================
# LIB B. Activation recorder
# =====================================================================
# We measure the residual-stream hidden state at each block output, image
# tokens only, from the conditional row of the guidance batch, cast to fp32
# on the GPU before any reduction.
#
# The image stream is picked by shape, not by tuple position. In a tuple
# output, None entries are skipped (an SD3-style final block returns
# (None, hidden_states)) and the 3-dim tensor with seq_len n_img is used.
# PixArt blocks return a plain tensor with seq_len n_img, since text enters
# through cross-attention. A plain tensor of length n_txt + n_img would have
# its leading text tokens sliced off. PixArt never produces that case.
#
# With classifier-free guidance, diffusers runs one batch-2 forward per step
# with rows [negative, positive]. The recorder keeps row `select_row`
# (1, the conditional row, checked in the smoke test). With guidance off
# the batch is 1 and row 0 is used. Any other output raises inside the
# hook. Hook errors are logged and never stop the generation.
import math
import numpy as np
import torch

STAT_NAMES = ("signed_mean", "mean_abs", "max_abs", "p99_abs", "frac_pos")

def enumerate_model_blocks(transformer):
    """List of (index, module, kind) in forward order."""
    return [(i, m, "block")
            for i, m in enumerate(transformer.transformer_blocks)]

def compute_channel_stats(X):
    """Per-channel statistics over tokens. X is a float32 [N_tokens, D]
    tensor on any device. Returns a dict of [D] float32 tensors.

    signed_mean    mean_n X[n,d], its absolute value is the default ranking statistic
    mean_abs       mean_n |X[n,d]|, used for the landscape
    max_abs        max_n |X[n,d]|, sensitive to single-token spikes
    p99_abs        99th percentile of |X[.,d]|, less spike-sensitive
    frac_pos       fraction of tokens with X[n,d] > 0, shows sign structure
    """
    N = X.shape[0]
    absX = X.abs()
    k99 = min(N, max(1, int(math.ceil(0.99 * N))))
    return {
        "signed_mean": X.mean(dim=0),
        "mean_abs":    absX.mean(dim=0),
        "max_abs":     absX.amax(dim=0),
        "p99_abs":     absX.kthvalue(k99, dim=0).values,
        "frac_pos":    (X > 0).to(torch.float32).mean(dim=0),
    }

class ActivationRecorder:
    def __init__(self, transformer, n_steps, n_img, n_txt, d_model, stat_blocks,
                 lstar=None, tstar=None, capture_full=False,
                 capture_token_norms=False, expected_batch=1, select_row=0):
        self.all_blocks = enumerate_model_blocks(transformer)
        self.n_steps, self.n_img, self.n_txt, self.d_model = \
            int(n_steps), int(n_img), int(n_txt), int(d_model)
        self.expected_batch = int(expected_batch)
        self.select_row = int(select_row)
        self.stat_blocks = sorted({int(b) for b in stat_blocks})
        bad = [b for b in self.stat_blocks if not (0 <= b < len(self.all_blocks))]
        if bad:
            raise ValueError(f"stat_blocks out of range: {bad} "
                             f"(model has {len(self.all_blocks)} blocks)")
        self.lstar = None if lstar is None else int(lstar)
        self.tstar = None if tstar is None else int(tstar)
        self.capture_full = bool(capture_full)
        self.capture_token_norms = bool(capture_token_norms)
        if self.capture_full or self.capture_token_norms:
            assert self.lstar in self.stat_blocks, \
                "capture requested but lstar not among stat_blocks"
        self._handles = []
        self.reset()

    def reset(self):
        self.stats = {b: {s: np.zeros((self.n_steps, self.d_model), np.float32)
                          for s in STAT_NAMES} for b in self.stat_blocks}
        self.counters = {b: 0 for b in self.stat_blocks}
        self.overflow = {b: 0 for b in self.stat_blocks}
        self.token_norms = (np.zeros((self.n_steps, self.n_img), np.float16)
                            if (self.capture_token_norms and self.lstar is not None)
                            else None)
        self.full_tensor = None
        self.full_clamped = 0
        self.errors = []

    # ------------------------------------------------ stream extraction
    def _extract_image_stream(self, output):
        def pick(t3):
            b = t3.shape[0]
            if b == self.expected_batch:
                return t3[self.select_row if b > 1 else 0]
            if self.expected_batch == 2 and b == 1:   # guidance disabled
                return t3[0]
            raise RuntimeError(f"unexpected batch size {b} "
                               f"(expected {self.expected_batch})")
        if isinstance(output, (tuple, list)):
            cands = [o for o in output
                     if o is not None and torch.is_tensor(o) and o.dim() == 3]
            for o in cands:
                if o.shape[1] == self.n_img:
                    return pick(o)
            raise RuntimeError(
                "no image stream (seq_len=%d) in tuple output; got shapes %s"
                % (self.n_img, [tuple(o.shape) for o in cands]))
        if torch.is_tensor(output) and output.dim() == 3:
            L = output.shape[1]
            if L == self.n_img:
                return pick(output)
            if self.n_txt and L == self.n_img + self.n_txt:
                return pick(output)[self.n_txt:, :]
            raise RuntimeError(
                f"unexpected sequence length {L} "
                f"(expected {self.n_img} or {self.n_txt + self.n_img})")
        raise RuntimeError(f"unexpected hook output type {type(output)}")

    # ------------------------------------------------------------ hooks
    def _make_hook(self, bidx):
        def hook(module, args, output):
            step = self.counters[bidx]
            self.counters[bidx] = step + 1
            if step >= self.n_steps:
                self.overflow[bidx] += 1
                return
            try:
                X = self._extract_image_stream(output)
                if X.shape[0] != self.n_img:
                    raise RuntimeError(f"token count {X.shape[0]} != n_img {self.n_img}")
                if X.shape[1] != self.d_model:
                    raise RuntimeError(f"hidden size {X.shape[1]} != d_model {self.d_model}")
                X = X.to(torch.float32)
                st = compute_channel_stats(X)
                packed = torch.stack([st[s] for s in STAT_NAMES], dim=0) \
                              .to("cpu", torch.float32).numpy()   # one device-to-host copy
                for si, s in enumerate(STAT_NAMES):
                    self.stats[bidx][s][step] = packed[si]
                if self.token_norms is not None and bidx == self.lstar:
                    self.token_norms[step] = \
                        X.norm(dim=1).to("cpu", torch.float16).numpy()
                if (self.capture_full and bidx == self.lstar
                        and self.tstar is not None and step == self.tstar):
                    n_over = int((X.abs() > 65504.0).sum().item())
                    if n_over:                      # would overflow fp16, so clamp and count
                        self.full_clamped = n_over
                    self.full_tensor = X.clamp(-65504.0, 65504.0) \
                                        .to("cpu", torch.float16).numpy()
            except Exception as e:   # log it, don't crash the generation
                self.errors.append(
                    f"block {bidx} step {step}: {type(e).__name__}: {e}")
        return hook

    def attach(self):
        assert not self._handles, "recorder already attached"
        hooked = set(self.stat_blocks)
        for i, mod, _kind in self.all_blocks:
            if i in hooked:
                self._handles.append(mod.register_forward_hook(self._make_hook(i)))
        return self

    def detach(self):
        for h in self._handles:
            h.remove()
        self._handles = []

    def finalize(self, expected_steps=None):
        exp = int(self.n_steps if expected_steps is None else expected_steps)
        fires_ok = (all(self.counters[b] == exp for b in self.stat_blocks)
                    and all(v == 0 for v in self.overflow.values()))
        nan_flag = False
        for b in self.stat_blocks:
            for s in STAT_NAMES:
                if not np.isfinite(self.stats[b][s][:exp]).all():
                    nan_flag = True
        return {"fires_ok": bool(fires_ok),
                "nan_flag": bool(nan_flag),
                "hook_errors": len(self.errors),
                "errors": list(self.errors),
                "counters": {int(b): int(c) for b, c in self.counters.items()},
                "tensor_captured": self.full_tensor is not None,
                "full_tensor_clamped": int(self.full_clamped)}

print(f"[lib-recorder] loaded (stats: {', '.join(STAT_NAMES)})")


In [ ]:
# =====================================================================
# LIB C. Scenario runner
# =====================================================================
# run_scenario does one generation with the recorder attached and saves the
# results atomically. The smoke test, the pilot (phase A) and the main run
# (phase B) all call it. Files are saved as png, then tensor, then npz. The
# npz comes last and marks the scenario as done, so a scenario that was cut
# off halfway is simply run again.
import gc, time, datetime
import numpy as np
import torch

def run_scenario(pipe, cfg, sid, pid, prompt_text, seed, embeds, out_dir,
                 stat_blocks, n_steps, n_img, pipe_dtype, device="cuda",
                 lstar=None, tstar=None, primary_stat=None,
                 capture_full=False, capture_token_norms=False,
                 tag="main", extra_meta=None, force=False):
    extra_meta = dict(extra_meta or {})
    need_tensor = bool(capture_full and cfg.save_full_tensor)
    paths = scenario_paths(out_dir, sid, need_tensor, lstar, tstar)
    expect = None
    if lstar is not None and tstar is not None and primary_stat is not None:
        expect = {"lstar": int(lstar), "tstar": int(tstar),
                  "primary_stat": primary_stat}
    if not force and is_scenario_complete(paths, need_tensor=need_tensor,
                                          expect=expect):
        return {"tag": tag, "sid": sid, "prompt_id": pid, "seed": seed,
                "status": "skipped",
                "precision_mode": extra_meta.get("precision_mode", "")}

    E = embeds[pid]
    n_txt = int(cfg.n_txt_in_seq)
    exp_batch = 2 if (cfg.guidance_mode == "cfg_batch2"
                      and cfg.guidance_scale > 1.0) else 1
    rec = ActivationRecorder(
        pipe.transformer, n_steps=n_steps, n_img=n_img, n_txt=n_txt,
        d_model=cfg.d_model, stat_blocks=stat_blocks,
        lstar=lstar, tstar=tstar,
        capture_full=need_tensor, capture_token_norms=capture_token_norms,
        expected_batch=exp_batch, select_row=cfg.cond_row)

    torch.manual_seed(seed)                         # global RNG as well
    gen = torch.Generator("cpu").manual_seed(seed)  # CPU generator gives the same noise on any device
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    rec.attach()
    try:
        with torch.inference_mode():
            out = pipe(prompt_embeds=E["pe"].to(device, pipe_dtype),
                       prompt_attention_mask=E["pm"].to(device),
                       negative_prompt_embeds=E["ne"].to(device, pipe_dtype),
                       negative_prompt_attention_mask=E["nm"].to(device),
                       # the default negative_prompt is "", which clashes
                       # with passing precomputed negative embeds
                       negative_prompt=None,
                       height=cfg.height, width=cfg.width,
                       num_inference_steps=n_steps,
                       guidance_scale=cfg.guidance_scale,
                       generator=gen, output_type="pil",
                       use_resolution_binning=False)
    finally:
        rec.detach()
    wall = time.time() - t0
    peak = (torch.cuda.max_memory_allocated() / 2**30
            if torch.cuda.is_available() else 0.0)
    image = out.images[0]
    fin = rec.finalize(expected_steps=n_steps)
    if fin["errors"]:
        append_error_log(ERROR_LOG_PATH, sid,
                         "hook errors:\n  " + "\n  ".join(fin["errors"][:20]))

    sig = (pipe.scheduler.sigmas.detach().float().cpu().numpy()
           if hasattr(pipe.scheduler, "sigmas") else np.zeros(0, np.float32))
    tsv = (pipe.scheduler.timesteps.detach().float().cpu().numpy()
           if hasattr(pipe.scheduler, "timesteps") else np.zeros(0, np.float32))

    arrays = {}
    for b in rec.stat_blocks:
        for s in STAT_NAMES:
            arrays[f"b{b:02d}__{s}"] = rec.stats[b][s]
    if rec.token_norms is not None:
        arrays["token_norms_lstar"] = rec.token_norms

    diag = {"tag": tag, "sid": sid, "prompt_id": pid, "seed": seed,
            "wall_s": round(wall, 1), "peak_vram_gib": round(peak, 2),
            "fires_ok": fin["fires_ok"], "nan_flag": fin["nan_flag"],
            "hook_errors": fin["hook_errors"],
            "tensor_captured": fin["tensor_captured"],
            "precision_mode": extra_meta.get("precision_mode", "")}

    # ---- per-scenario top-k under the primary statistic and under mean|x|, and their overlap
    if lstar is not None and tstar is not None and primary_stat is not None:
        stats_t = {s: rec.stats[lstar][s][tstar] for s in STAT_NAMES}
        sp = primary_score(stats_t, primary_stat)
        sa = np.asarray(stats_t["mean_abs"], dtype=np.float64)
        ids_p = topk_ids(sp, cfg.top_save)
        ids_a = topk_ids(sa, cfg.top_save)
        arrays["top20_ids"] = ids_p.astype(np.int32)
        arrays["top20_scores"] = sp[ids_p].astype(np.float32)
        arrays["top20_ids_mean_abs"] = ids_a.astype(np.int32)
        arrays["top20_scores_mean_abs"] = sa[ids_a].astype(np.float32)
        cons = sign_consistency(stats_t["frac_pos"])
        top1 = int(ids_p[0])
        med = max(float(np.median(sp)), 1e-12)
        diag.update(
            top1_channel=top1,
            top1_score=round(float(sp[top1]), 4),
            top1_over_median=round(float(sp[top1] / med), 1),
            top1_sign_consistency=round(float(cons[top1]), 4),
            jacc_primary_vs_meanabs_top10=round(jaccard(ids_p[:10], ids_a[:10]), 3))

    meta = {"sid": sid, "prompt_id": pid, "seed": int(seed), "prompt": prompt_text,
            "tag": tag, "model_id": cfg.model_id,
            "height": cfg.height, "width": cfg.width, "num_steps": int(n_steps),
            "guidance_scale": cfg.guidance_scale,
            "guidance_mode": cfg.guidance_mode, "cond_row": int(cfg.cond_row),
            "n_img": int(n_img), "n_txt": int(n_txt), "d_model": cfg.d_model,
            "stat_blocks": [int(b) for b in rec.stat_blocks],
            "lstar": lstar, "tstar": tstar, "primary_stat": primary_stat,
            "counters": fin["counters"], "fires_ok": fin["fires_ok"],
            "nan_flag": fin["nan_flag"], "hook_errors": fin["hook_errors"],
            "hook_error_msgs": fin["errors"][:10],
            "tensor_captured": fin["tensor_captured"],
            "full_tensor_clamped": fin["full_tensor_clamped"],
            "wall_s": round(wall, 1), "peak_vram_gib": round(peak, 2),
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            **extra_meta}
    arrays["sigmas"] = sig.astype(np.float32)
    arrays["timesteps"] = tsv.astype(np.float32)
    arrays["meta_json"] = meta_to_npz_field(meta)

    save_png_atomic(image, paths["png"])
    if need_tensor and rec.full_tensor is not None:
        save_npy_atomic(paths["tensor"], rec.full_tensor)
    save_npz_atomic(paths["npz"], **arrays)   # written last, marks completion

    ok = (fin["fires_ok"] and not fin["nan_flag"] and fin["hook_errors"] == 0
          and (fin["tensor_captured"] or not need_tensor))
    diag["status"] = "ok" if ok else "warn"
    return diag

print("[lib-run] run_scenario ready")


In [ ]:
# =====================================================================
# S3c. Smoke test
# =====================================================================
# First we check the block count. Then we check that the row we treat as
# conditional under classifier-free guidance matches a guidance-free run at
# step 0, where the input latents are the same, and matches it better than
# the other row does. Last comes one short generation through run_scenario.
# It has to find the image stream (4096 tokens) in every block, fire each
# hook exactly `smoke_steps` times without errors, give finite stats, and
# save a tensor and token norms that load back correctly.
print("=" * 70)
print("SMOKE TEST")
print("=" * 70)

_blocks = enumerate_model_blocks(pipe.transformer)
assert len(_blocks) == CFG.n_blocks_expected, (
    f"block census mismatch: found {len(_blocks)} blocks, expected "
    f"{CFG.n_blocks_expected}. diffusers {LIB_VERSIONS.get('diffusers')} "
    f"may have restructured the model -- inspect pipe.transformer before "
    f"proceeding.")
print(f"[smoke] block census OK: {len(_blocks)} blocks")

# ---- which row of the guidance batch is conditional
_probe_block = int(CFG.block_tie_priority[0])
_probe_mod = _blocks[_probe_block][1]

class _RowProbe:
    """Record the per-channel mean |x| of one batch row at step 0."""
    def __init__(self, row):
        self.row, self.v, self.fired = row, None, 0
    def hook(self, module, args, output):
        if self.fired == 0:
            o = output
            if isinstance(o, (tuple, list)):
                o = [t for t in o if t is not None and torch.is_tensor(t)
                     and t.dim() == 3 and t.shape[1] == N_IMG][0]
            r = self.row if o.shape[0] > 1 else 0
            self.v = o[r].to(torch.float32).abs().mean(dim=0).cpu().numpy()
        self.fired += 1

def _short_run(guidance, probes, steps=2):
    E = EMBEDS["p00"]
    hs = [_probe_mod.register_forward_hook(p.hook) for p in probes]
    try:
        with torch.inference_mode():
            _ = pipe(prompt_embeds=E["pe"].to(DEVICE, DTYPE),
                     prompt_attention_mask=E["pm"].to(DEVICE),
                     negative_prompt_embeds=E["ne"].to(DEVICE, DTYPE),
                     negative_prompt_attention_mask=E["nm"].to(DEVICE),
                     negative_prompt=None,
                     height=CFG.height, width=CFG.width,
                     num_inference_steps=steps, guidance_scale=guidance,
                     generator=torch.Generator("cpu").manual_seed(0),
                     output_type="latent",
                     use_resolution_binning=False)
    finally:
        for h in hs:
            h.remove()

_p_cond = _RowProbe(CFG.cond_row)
_p_unc = _RowProbe(1 - CFG.cond_row)
_p_ref = _RowProbe(0)
_short_run(CFG.guidance_scale, [_p_cond, _p_unc])
_short_run(1.0, [_p_ref])                     # guidance off, so batch 1

def _corr(a, b):
    a = np.asarray(a, np.float64); b = np.asarray(b, np.float64)
    a = (a - a.mean()) / max(a.std(), 1e-12)
    b = (b - b.mean()) / max(b.std(), 1e-12)
    return float((a * b).mean())

_r_cond = _corr(_p_cond.v, _p_ref.v)
_r_unc = _corr(_p_unc.v, _p_ref.v)
print(f"[smoke] conditional-row check at block {_probe_block}, step 0: "
      f"corr(selected row, guidance-free) = {_r_cond:.4f} | "
      f"corr(other row, guidance-free) = {_r_unc:.4f}")
assert _r_cond > 0.98 and _r_cond > _r_unc, (
    "conditional-row selection failed validation: the selected row does "
    "not reproduce the guidance-free statistics at step 0. Check "
    "CFG.cond_row against the diffusers concatenation order "
    "[negative, positive] (conditional = index 1).")
print("[smoke] conditional-row selection validated (row "
      f"{CFG.cond_row} of the CFG batch)")

# ---- short generation with all hooks, through run_scenario
SMOKE_SID = "smoke__p00_s0"
SMOKE_LSTAR, SMOKE_TSTAR = _probe_block, CFG.smoke_steps - 1
for _f in glob.glob(os.path.join(PATHS["smoke"], "smoke__*")):
    os.remove(_f)                       # always a fresh smoke run

_diag = run_scenario(
    pipe, CFG, SMOKE_SID, "p00", PROMPTS["p00"], 0, EMBEDS, PATHS["smoke"],
    stat_blocks=list(range(len(_blocks))), n_steps=CFG.smoke_steps, n_img=N_IMG,
    pipe_dtype=DTYPE, device=DEVICE, lstar=SMOKE_LSTAR, tstar=SMOKE_TSTAR,
    primary_stat=CFG.primary_stat_default,
    capture_full=True, capture_token_norms=True, tag="smoke",
    extra_meta={"precision_mode": PRECISION_MODE, "config_hash": CONFIG_HASH},
    force=True)
print(f"[smoke] generation done in {_diag['wall_s']}s "
      f"(peak {_diag['peak_vram_gib']} GiB) status={_diag['status']}")
assert _diag["status"] == "ok", f"smoke diagnostics not clean: {_diag}"

_paths = scenario_paths(PATHS["smoke"], SMOKE_SID, True, SMOKE_LSTAR, SMOKE_TSTAR)
with np.load(_paths["npz"], allow_pickle=False) as _z:
    _meta = meta_from_npz(_z)
    assert _meta["n_img"] == N_IMG, f"n_img={_meta['n_img']}"
    assert all(int(c) == CFG.smoke_steps for c in _meta["counters"].values()), \
        f"hook fire counts wrong: {_meta['counters']}"
    _stat_keys = [k for k in _z.files if k.startswith("b") and "__" in k]
    assert len(_stat_keys) == len(_blocks) * len(STAT_NAMES), \
        f"{len(_stat_keys)} stat arrays, expected {len(_blocks)*len(STAT_NAMES)}"
    _tn = _z["token_norms_lstar"]
    assert _tn.shape == (CFG.smoke_steps, N_IMG) and _tn.dtype == np.float16
    assert _z["sigmas"].shape[0] in (0, CFG.smoke_steps, CFG.smoke_steps + 1)
    for _k in _stat_keys:
        assert np.isfinite(_z[_k]).all(), f"non-finite values in {_k}"

_X = np.load(_paths["tensor"])
assert _X.shape == (N_IMG, CFG.d_model) and _X.dtype == np.float16, \
    f"tensor round-trip: shape {_X.shape} dtype {_X.dtype}"
_Xf = _X.astype(np.float32)
assert np.isfinite(_Xf).all() and float(np.abs(_Xf).max()) > 0.0
del _X, _Xf

from PIL import Image
fig, ax = plt.subplots(figsize=(2.6, 2.6))
ax.imshow(Image.open(_paths["png"])); ax.axis("off")
ax.set_title(f"smoke image, {CFG.smoke_steps} steps (low quality expected)",
             fontsize=7)
plt.show()
print("SMOKE TEST PASSED -- safe to start Phase A (pilot).")


## S4. Phase A, pilot capture

Five scenarios that cover both prompts and seeds, each generated over the
full trajectory, with reduced statistics logged at every block. Per-token
norms are also saved at the probe block (13) for the position check in S5.
The pilot data has to be clean, so a warning here stops the cell instead of
feeding into the decision.


In [ ]:
# =====================================================================
# S4. Phase A, pilot capture
# =====================================================================
# Five scenarios, all 28 blocks, full 20-step generations, resumable.
# Per-token L2 norms are also saved at the probe block
# (CFG.anchor_probe_block) for the position check in S5. Pilot files get
# the prefix pilot__ and live apart from the main run, since they log
# every block instead of l* and the secondary blocks.

print("=" * 70)
print(f"PHASE A - PILOT ({len(CFG.pilot_pairs)} scenarios, all "
      f"{CFG.n_blocks_expected} blocks, {CFG.num_steps} steps each)")
print("=" * 70)

PILOT_SIDS = []
for _pid, _seed in CFG.pilot_pairs:
    _sid = f"pilot__{scenario_id(_pid, _seed)}"
    PILOT_SIDS.append(_sid)
    _diag = run_scenario(
        pipe, CFG, _sid, _pid, PROMPTS[_pid], _seed, EMBEDS, PATHS["pilot"],
        stat_blocks=list(range(CFG.n_blocks_expected)),
        n_steps=CFG.num_steps, n_img=N_IMG, pipe_dtype=DTYPE, device=DEVICE,
        lstar=CFG.anchor_probe_block, tstar=None, primary_stat=None,
        capture_full=False, capture_token_norms=True, tag="pilot",
        extra_meta={"precision_mode": PRECISION_MODE, "config_hash": CONFIG_HASH})
    if _diag["status"] != "skipped":
        append_run_log(RUN_LOG_PATH, _diag)
    print(f"  {_sid:22s} {_diag['status']:8s} "
          f"({_diag.get('wall_s', 0)}s)  fires_ok={_diag.get('fires_ok', '-')} "
          f"nan={_diag.get('nan_flag', '-')}")
    if _diag["status"] == "warn":
        raise RuntimeError(
            f"pilot scenario {_sid} produced warnings: {_diag} -- "
            f"pilot data must be clean; investigate before the decision cell.")
print("[pilot] Phase A capture complete.")


## S5. Pilot analysis and decision (P1 to P6)

This cell applies the rules that were fixed before the pilot and writes the
result to `pilot_decision.json`. Phase B won't run without that file.

P1 picks the analysis block l\*. For each block it measures how concentrated
the channels are, as the log ratio of the top-1 to the median mean absolute
activation and as an effective number of dominant channels. The band block
with the highest median log ratio over the pilots wins, and near-ties follow
the configured tie priority. Its top-2 channels also have to be the same in
every pilot. If they aren't, the block is dropped and the selection runs
again.

P2 plots the sorted channel profiles, where the massive channels show up as
a cliff.

P3 picks the analysis step t\*. The default is the middle of the trajectory.
It is kept if the top-channel magnitude there is at least 60 percent of the
final-step value and the top-10 channels overlap the final step with a
Jaccard index of at least 0.6. Otherwise the step at three quarters of the
trajectory is used.

P4 compares the rankings by absolute signed mean and by mean absolute value.
If they agree poorly, or if a massive channel has mixed sign, the primary
statistic becomes the mean absolute value before Phase B, and this is
recorded as a caveat on comparability.

P5 is a pipeline check based on the model itself. PixArt-Sigma puts its
high-norm token at the bottom right of the image for every prompt and seed.
So in every pilot, the max-norm token at the probe block has to lie in the
bottom-right 8 by 8 corner of the 64 by 64 token grid. If it doesn't, the
cell stops and prints what to check. An override flag exists but has to be
set by hand.

P6 counts how often channels recur in the top 10 across the five pilots.

In [ ]:
# =====================================================================
# LIB D. Pilot decision functions
# =====================================================================
# Plain numpy, so these can be unit tested. S5 uses them for these rules.
#   P1, P2  block concentration and choice of l*, with band, tie priority
#           and tie window from S1. The top-2 channels must match in
#           every pilot.
#   P3      choice of t*, default step unless magnitude or identity
#           stability is too low
#   P4      agreement between |mean| and mean|x|, which fixes the primary
#           statistic before Phase B
#   P5      ranks of externally reported anchor channels, for models
#           where such channels are known
#   P6      recurrence check across the five pilots
import os, json, math
import numpy as np

def open_pilot_npzs(pilot_dir, sids):
    return {sid: np.load(os.path.join(pilot_dir, f"{sid}.npz"),
                         allow_pickle=False) for sid in sids}

def get_stat(z, block, stat):
    """[n_steps, D] array for one block/statistic from a scenario npz."""
    return z[f"b{block:02d}__{stat}"]

def block_concentration_table(zs, blocks, step):
    """For each block and pilot, log10(top1 / median) of mean|x| and the
    effective number of dominant channels (sum s^2)^2 / sum s^4, which is 1
    for a single spike."""
    out = {}
    for b in blocks:
        ratios, effs = [], []
        for z in zs.values():
            v = get_stat(z, b, "mean_abs")[step].astype(np.float64)
            med = max(float(np.median(v)), 1e-12)
            ratios.append(math.log10(max(float(v.max()), 1e-12) / med))
            s2 = v ** 2
            effs.append(float((s2.sum() ** 2) / max((s2 ** 2).sum(), 1e-24)))
        out[int(b)] = {"log10_top1_over_median": ratios,
                       "eff_num_dominant": effs}
    return out

def choose_block(conc, band, tie_priority, tie_window):
    med = {b: float(np.median(v["log10_top1_over_median"]))
           for b, v in conc.items() if band[0] <= b <= band[1]}
    if not med:
        raise RuntimeError("no candidate blocks in band")
    best = max(med.values())
    ties = sorted(b for b, m in med.items() if m >= best - tie_window)
    for pref in tie_priority:
        if pref in ties:
            return int(pref), med, ties
    return int(max(med, key=med.get)), med, ties

def top2_stable(zs, block, step):
    sets = [frozenset(int(i) for i in
                      topk_ids(get_stat(z, block, "mean_abs")[step], 2))
            for z in zs.values()]
    return len(set(sets)) == 1, [sorted(s) for s in sets]

def choose_block_with_stability(zs, conc, cfg, step):
    """l* is the most concentrated band block whose top-2 channels are the
    same in every pilot. A block that fails this is dropped and the
    selection runs again."""
    excluded = []
    while True:
        sub = {b: v for b, v in conc.items()
               if cfg.band[0] <= b <= cfg.band[1] and b not in excluded}
        if not sub:
            raise RuntimeError(
                "No block in the band passes the top-2 stability check. "
                "Inspect the pilot npz files manually before proceeding.")
        b, med, ties = choose_block(sub, cfg.band, cfg.block_tie_priority,
                                    cfg.block_tie_window_log10)
        ok, sets = top2_stable(zs, b, step)
        if ok:
            return b, med, ties, excluded, sets
        excluded.append(b)

def tstar_check(zs, block, cfg):
    last = cfg.num_steps - 1
    m_def = np.array([float(get_stat(z, block, "mean_abs")[cfg.t_star_default].max())
                      for z in zs.values()])
    m_last = np.array([float(get_stat(z, block, "mean_abs")[last].max())
                       for z in zs.values()])
    frac = float(np.median(m_def / np.maximum(m_last, 1e-12)))
    jaccs = []
    for z in zs.values():
        vt = get_stat(z, block, "mean_abs")[cfg.t_star_default]
        vl = get_stat(z, block, "mean_abs")[last]
        jaccs.append(jaccard(topk_ids(vt, 10), topk_ids(vl, 10)))
    stab = float(np.median(jaccs))
    ok = (frac >= cfg.tstar_mag_frac_min) and (stab >= cfg.tstar_stab_jaccard_min)
    t_star = int(cfg.t_star_default if ok else cfg.t_star_fallback)
    return t_star, {"mag_frac_median": frac,
                    "stability_jaccard_median": stab,
                    "per_pilot_jaccard": [float(j) for j in jaccs],
                    "passed_default": bool(ok),
                    "t_default": int(cfg.t_star_default),
                    "t_fallback": int(cfg.t_star_fallback)}

def stat_gate(zs, block, t, cfg):
    """P4 gate, run on the pilot. If it passes, the primary statistic stays
    |mean|. It fails when agreement is
    poor or when a massive channel has mixed sign that |mean| would cancel.
    Then the primary statistic becomes mean|x| and the decision file notes
    the caveat."""
    jaccs, mixed = [], []
    for sid, z in zs.items():
        sm = np.abs(get_stat(z, block, "signed_mean")[t].astype(np.float64))
        ma = get_stat(z, block, "mean_abs")[t].astype(np.float64)
        cons = sign_consistency(get_stat(z, block, "frac_pos")[t])
        ids_m, ids_a = topk_ids(sm, 10), topk_ids(ma, 10)
        jaccs.append(jaccard(ids_m, ids_a))
        for ch in ids_a:
            if cons[int(ch)] < cfg.sign_consistency_min:
                mixed.append({"sid": sid, "channel": int(ch),
                              "consistency": round(float(cons[int(ch)]), 4),
                              "mean_abs": round(float(ma[int(ch)]), 4)})
    med_j = float(np.median(jaccs))
    passed = (med_j >= cfg.stat_gate_jaccard_min) and (len(mixed) == 0)
    return {"median_jaccard": med_j,
            "per_pilot_jaccard": [float(j) for j in jaccs],
            "mixed_sign_hits": mixed,
            "passed": bool(passed),
            "primary_stat": (cfg.primary_stat_default if passed else "mean_abs")}

def anchor_min_ranks(zs, blocks, t, anchor):
    """Best rank by mean|x| of channel `anchor` over the pilots, as {block: rank}."""
    out = {}
    for b in blocks:
        out[int(b)] = min(rank_of(get_stat(z, b, "mean_abs")[t], anchor)
                          for z in zs.values())
    return out

def anchor_check(zs, band_blocks, t, anchors, top_rank=20):
    detail = {}
    for a in anchors:
        per_block = anchor_min_ranks(zs, band_blocks, t, int(a))
        b_best = min(per_block, key=per_block.get)
        detail[int(a)] = {"min_rank": int(per_block[b_best]),
                          "best_block": int(b_best),
                          "per_block": per_block}
    found = any(d["min_rank"] <= top_rank for d in detail.values())
    return {"found": bool(found), "top_rank": int(top_rank), "detail": detail}

def mini_recurrence(zs, block, t, primary, k=10):
    counts = {}
    for z in zs.values():
        stats_t = {s: get_stat(z, block, s)[t] for s in STAT_NAMES}
        for ch in topk_ids(primary_score(stats_t, primary), k):
            counts[int(ch)] = counts.get(int(ch), 0) + 1
    n = len(zs)
    return {ch: c / n for ch, c in
            sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))}

print("[lib-decision] loaded (P1-P6 rules)")


In [ ]:
# =====================================================================
# S5. Pilot analysis and decision (P1 to P6)
# =====================================================================
# Applies the selection rules to the pilot data and writes
# pilot_decision.json, which Phase B needs. Figures are saved as PDF and
# PNG with a draft caption.
print("=" * 70); print("S5 - PILOT ANALYSIS & DECISION"); print("=" * 70)
zs = open_pilot_npzs(PATHS["pilot"], PILOT_SIDS)
ALL_BLOCKS = list(range(CFG.n_blocks_expected))
STEP0 = CFG.t_star_default
conc = block_concentration_table(zs, ALL_BLOCKS, STEP0)

# -- P1 concentration by block -------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(FIG_PAGE, 2.3))
for j, (key, ylab, logy) in enumerate(
        [("log10_top1_over_median",
          r"$\log_{10}$ top-1 / median of mean $|x|$", False),
         ("eff_num_dominant", "effective dominant channels", True)]):
    per = np.array([[conc[b][key][i] for b in ALL_BLOCKS]
                    for i in range(len(zs))])
    ax = axes[j]
    for i in range(per.shape[0]):
        ax.plot(ALL_BLOCKS, per[i], lw=0.6, color="0.7")
    ax.plot(ALL_BLOCKS, np.median(per, axis=0), lw=1.6,
            color=OKABE_ITO[0], label="median of pilots")
    ax.axvspan(CFG.band[0], CFG.band[1], color=OKABE_ITO[1], alpha=0.12,
               lw=0, label=f"search band {CFG.band[0]} to {CFG.band[1]}")
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel(BLOCK_AXIS_LABEL)
    ax.set_ylabel(ylab)
    legend_above(ax, ncol=2)
save_fig(fig, fig_stem("P1_block_concentration"), caption=(
    f"Channel concentration by block in {MODEL_LABEL} at denoising step "
    f"{STEP0}. Left: log ratio of the largest to the median per-channel "
    f"mean absolute activation. Right: effective number of dominant "
    f"channels. Thin lines are the {len(zs)} pilot scenarios, the thick "
    f"line their median; the shaded region is the pre-registered search "
    f"band."))

# -- block choice and top-2 stability across pilots -----------------
L_STAR, med_ratio, ties, excluded, top2sets = \
    choose_block_with_stability(zs, conc, CFG, STEP0)
print(f"[decision] l* = {L_STAR}  (median log10 ratio "
      f"{med_ratio[L_STAR]:.2f}; tie set {ties}; excluded-for-instability "
      f"{excluded}; pilot top-2 sets {top2sets})")
conc_fb = block_concentration_table(
    zs, [b for b in ALL_BLOCKS if CFG.band[0] <= b <= CFG.band[1]],
    CFG.t_star_fallback)
_b_fb, _, _ = choose_block(conc_fb, CFG.band, CFG.block_tie_priority,
                           CFG.block_tie_window_log10)
if _b_fb != L_STAR:
    print(f"[decision][note] at fallback step {CFG.t_star_fallback} the argmax "
          f"block would be {_b_fb} (chosen l*={L_STAR} at step {STEP0}); "
          f"recorded in the decision file, selection unchanged (pre-registered).")

# -- P2 sorted channel profiles -------------------------------------
cands = sorted({L_STAR, *CFG.block_tie_priority, CFG.band[0], CFG.band[1]})
fig, ax = plt.subplots(figsize=(FIG_COL, 2.2))
for j, b in enumerate(cands):
    prof = np.median(np.stack(
        [np.sort(get_stat(z, b, "mean_abs")[STEP0].astype(np.float64))[::-1]
         for z in zs.values()]), axis=0)
    ax.semilogy(prof, lw=1.0, color=OKABE_ITO[j % len(OKABE_ITO)],
                label=f"block {b}" + (" (selected)" if b == L_STAR else ""))
ax.set_xlabel("channel rank (descending)")
ax.set_ylabel(r"mean $|x|$ (median of pilots)")
legend_above(ax, ncol=2)
save_fig(fig, fig_stem("P2_sorted_profiles"), caption=(
    f"Sorted per-channel profiles of the mean absolute activation in "
    f"{MODEL_LABEL} at denoising step {STEP0} (median over the pilot "
    f"scenarios), for the selected block and reference blocks. The "
    f"isolated leading values form the massive-activation cliff."))

# -- P3 analysis step -----------------------------------------------
T_STAR, t_ev = tstar_check(zs, L_STAR, CFG)
last = CFG.num_steps - 1
fig, axes = plt.subplots(1, 2, figsize=(FIG_PAGE, 2.2))
for sid, z in zs.items():
    axes[0].plot(get_stat(z, L_STAR, "mean_abs").max(axis=1),
                 lw=0.7, color="0.6")
    vl = get_stat(z, L_STAR, "mean_abs")[last]
    axes[1].plot([jaccard(topk_ids(get_stat(z, L_STAR, "mean_abs")[s], 10),
                          topk_ids(vl, 10)) for s in range(CFG.num_steps)],
                 lw=0.7, color="0.6")
for ax, ylab in zip(axes, [r"$\max_d$ mean $|x|$",
                           "top-10 Jaccard vs final step"]):
    ax.axvline(CFG.t_star_default, color=OKABE_ITO[2], ls="--", lw=0.9,
               label=f"default step {CFG.t_star_default}")
    ax.axvline(CFG.t_star_fallback, color=OKABE_ITO[3], ls=":", lw=0.9,
               label=f"fallback step {CFG.t_star_fallback}")
    ax.set_xlabel("denoising step")
    ax.set_ylabel(ylab)
    legend_above(ax, ncol=2)
save_fig(fig, fig_stem("P3_step_evidence"), caption=(
    f"Evidence for the analysis-step rule in {MODEL_LABEL} at block "
    f"{L_STAR}. Left: magnitude of the largest channel across denoising "
    f"steps, per pilot scenario. Right: Jaccard overlap of the top-10 "
    f"channel set at each step with the final step. The default and "
    f"fallback steps of the pre-registered rule are marked."))
print(f"[decision] t* = {T_STAR}  (mag-frac {t_ev['mag_frac_median']:.2f} "
      f">= {CFG.tstar_mag_frac_min}? | identity-J {t_ev['stability_jaccard_median']:.2f} "
      f">= {CFG.tstar_stab_jaccard_min}? -> default passed = {t_ev['passed_default']})")

# -- P4 statistic agreement gate -----------------------------------
gate = stat_gate(zs, L_STAR, T_STAR, CFG)
PRIMARY_STAT = gate["primary_stat"]
print(f"[decision] P4 gate: median J(top10 |mean|, top10 mean|x|) = "
      f"{gate['median_jaccard']:.3f} (min {CFG.stat_gate_jaccard_min}); "
      f"mixed-sign hits = {len(gate['mixed_sign_hits'])} -> "
      f"{'PASS' if gate['passed'] else 'FAIL'} -> primary = {PRIMARY_STAT}")
if not gate["passed"]:
    print(f"[decision][CAVEAT] the two statistics disagree at "
          f"(l*={L_STAR}, t*={T_STAR}); primary flipped to mean|x|. Top-k "
          "lists are then NOT directly comparable to the Eq. 1 ranking of "
          "Turri et al.; this caveat is written into pilot_decision.json "
          "and the run manifest.")
    for h in gate["mixed_sign_hits"]:
        print(f"    mixed-sign channel: {h}")
    if not gate["mixed_sign_hits"]:
        print("[decision][note] P4 failed on top-10 tail agreement only "
              "(zero mixed-sign massive channels): the signed mean is not "
              "missing a massive channel; the flip to mean|x| is the "
              "conservative pre-registered response.")

# -- P5 position anchor check ---------------------------------------
# In PixArt-Sigma one high-norm token appears at the bottom right of the
# image for every prompt and seed. The pilot logged per-token norms at the
# probe block. In every pilot the max-norm token at t* has to fall in the
# bottom-right corner region of the token grid. No MA channels have been
# published for this model, so this check stands in for a channel anchor.
G = int(CFG.anchor_grid); R = int(CFG.anchor_region)
_detail = {}
fig, axes = plt.subplots(1, len(zs), figsize=(FIG_PAGE, 1.75))
for ax, (sid, z) in zip(np.atleast_1d(axes), zs.items()):
    tn = z["token_norms_lstar"][T_STAR].astype(np.float64).reshape(G, G)
    r, c = np.unravel_index(int(np.argmax(tn)), (G, G))
    ok = bool((r >= G - R) and (c >= G - R))
    _detail[sid] = {"row": int(r), "col": int(c), "ok": ok}
    ax.imshow(tn, cmap="viridis", interpolation="nearest")
    ax.scatter([c], [r], s=16, facecolors="none", edgecolors="w", lw=0.8)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(sid.replace("pilot__", ""), fontsize=6)
anch = {"kind": "internal_token_position",
        "found": all(d["ok"] for d in _detail.values()),
        "probe_block": int(CFG.anchor_probe_block), "t": int(T_STAR),
        "grid": G, "region": R, "detail": _detail}
save_fig(fig, fig_stem("P5_position_anchor"), caption=(
    f"Pipeline tripwire for PixArt-Sigma: per-token L2 norm of the block "
    f"{CFG.anchor_probe_block} output at the analysis step, reshaped to "
    f"the {G} by {G} latent token grid, for each pilot scenario. The "
    f"circle marks the maximum-norm token, which prior analysis of this "
    f"model places consistently at the bottom right of the image; the "
    f"check passes when it falls inside the bottom-right {R} by {R} "
    f"region."))
for sid, d in _detail.items():
    print(f"[decision] P5 position anchor {sid}: max-norm token at "
          f"(row {d['row']}, col {d['col']}) -> "
          f"{'OK' if d['ok'] else 'FAIL'}")
if not anch["found"] and not CFG.allow_missing_anchor:
    raise RuntimeError(
        "P5 TRIPWIRE: the maximum-norm token at the probe block is not in "
        "the bottom-right region in at least one pilot. Before proceeding, "
        "check: (1) the conditional-row selection under classifier-free "
        "guidance (diffusers concatenates [negative, positive], so the "
        "conditional row is index 1), (2) the row-major token raster "
        "assumption, (3) block indexing and hook placement on block "
        "outputs. Set CFG.allow_missing_anchor=True ONLY after manual "
        "verification.")

# -- P6 recurrence across pilots ------------------------------------
mini = mini_recurrence(zs, L_STAR, T_STAR, PRIMARY_STAT, k=10)
_always = [ch for ch, f in mini.items() if f == 1.0]
print(f"[decision] P6 mini-recurrence ({len(zs)} pilots, top-10 by "
      f"{PRIMARY_STAT}): {len(_always)} channel(s) in every pilot: {_always}")
print("             full table:",
      {ch: round(f, 2) for ch, f in list(mini.items())[:15]})

# -- write the decision ---------------------------------------------
DECISION = {
    "l_star": int(L_STAR), "t_star": int(T_STAR), "primary_stat": PRIMARY_STAT,
    "rules": {"band": list(CFG.band), "tie_priority": list(CFG.block_tie_priority),
              "tie_window_log10": CFG.block_tie_window_log10,
              "stat_gate_jaccard_min": CFG.stat_gate_jaccard_min,
              "sign_consistency_min": CFG.sign_consistency_min,
              "tstar_mag_frac_min": CFG.tstar_mag_frac_min,
              "tstar_stab_jaccard_min": CFG.tstar_stab_jaccard_min},
    "evidence": {
        "block_median_log10_ratio_band": {str(b): round(m, 3)
                                          for b, m in sorted(med_ratio.items())},
        "tie_set": ties, "excluded_unstable": excluded,
        "pilot_top2_sets": top2sets,
        "argmax_block_at_fallback_step": int(_b_fb),
        "tstar": t_ev, "stat_gate": gate, "anchor": anch,
        "mini_recurrence_top10": {str(k): v for k, v in list(mini.items())[:20]}},
    "pilot_sids": PILOT_SIDS,
    "precision_mode": PRECISION_MODE, "config_hash": CONFIG_HASH,
    "lib_versions": LIB_VERSIONS,
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds")}
save_json_atomic(DECISION, os.path.join(PATHS["meta"], "pilot_decision.json"))
for z in zs.values():
    z.close()

print()
print("=" * 70)
print("PILOT DECISION SUMMARY")
print("=" * 70)
print(f"  fixed block  l* = {L_STAR}   "
      f"(band {CFG.band}, tie-priority {CFG.block_tie_priority}; "
      f"median log10 top1/median = {med_ratio[L_STAR]:.2f})")
print(f"  fixed step   t* = {T_STAR}   "
      f"(mag-frac {t_ev['mag_frac_median']:.2f}, identity-J "
      f"{t_ev['stability_jaccard_median']:.2f})")
print(f"  primary stat    = {PRIMARY_STAT}   "
      f"(P4 gate {'PASS' if gate['passed'] else 'FAIL'}, "
      f"median J = {gate['median_jaccard']:.3f})")
_anchor_pos = {s.replace('pilot__', ''): (d['row'], d['col'])
               for s, d in anch['detail'].items()}
print(f"  position anchor = {anch['found']}  {_anchor_pos}")
print(f"  always-recurring in pilot top-10: {_always}")
print(f"  -> {os.path.join(PATHS['meta'], 'pilot_decision.json')}")
print("=" * 70)


## S5b. Optional check at 10 steps

The main run uses the standard 20 steps. This cell reruns the five pilot
scenarios at 10 steps and compares their top-10 channel sets with the
20-step pilot at the same relative step, using the Jaccard index. High
overlap means channel identity is the same in both settings. Low overlap
would be a reason to rerun the experiment at 10 steps. The result is added
to `pilot_decision.json`.

In [ ]:
# =====================================================================
# S5b. Optional check at 10 steps
# =====================================================================
# Reruns the five pilot scenarios at 10 steps and compares their top-10
# channels by mean|x| at l* with the 20-step pilot at the same relative
# step. The median Jaccard overlap is added to pilot_decision.json.

_dec_path = os.path.join(PATHS["meta"], "pilot_decision.json")
assert os.path.exists(_dec_path), "run the S5 decision cell first"
with open(_dec_path) as _f:
    DECISION = json.load(_f)
L_STAR = int(DECISION["l_star"]); T_STAR = int(DECISION["t_star"])

_bdir = os.path.join(PATHS["base"], "pilot_bridge")
os.makedirs(_bdir, exist_ok=True)
_tb = min(CFG.bridge_steps - 1,
          int(round(T_STAR / CFG.num_steps * CFG.bridge_steps)))
print(f"[bridge] {CFG.bridge_steps}-step reruns of the pilot pairs; "
      f"comparing top-10 at step {_tb} of {CFG.bridge_steps} vs "
      f"step {T_STAR} of {CFG.num_steps} (matched relative position)")

_jaccs = []
for _pid, _seed in CFG.pilot_pairs:
    _sid = f"bridge__{scenario_id(_pid, _seed)}"
    _diag = run_scenario(
        pipe, CFG, _sid, _pid, PROMPTS[_pid], _seed, EMBEDS, _bdir,
        stat_blocks=[L_STAR], n_steps=CFG.bridge_steps, n_img=N_IMG,
        pipe_dtype=DTYPE, device=DEVICE, lstar=None, tstar=None,
        primary_stat=None, capture_full=False, capture_token_norms=False,
        tag="bridge",
        extra_meta={"precision_mode": PRECISION_MODE,
                    "config_hash": CONFIG_HASH})
    with np.load(os.path.join(_bdir, f"{_sid}.npz"),
                 allow_pickle=False) as _zb, \
         np.load(os.path.join(PATHS["pilot"],
                              f"pilot__{scenario_id(_pid, _seed)}.npz"),
                 allow_pickle=False) as _zp:
        _t10b = topk_ids(_zb[f"b{L_STAR:02d}__mean_abs"][_tb], 10)
        _t10p = topk_ids(_zp[f"b{L_STAR:02d}__mean_abs"][T_STAR], 10)
    _j = jaccard(_t10b, _t10p)
    _jaccs.append(_j)
    print(f"    {_sid:26s} J(top-10) = {_j:.2f}")
_med = float(np.median(_jaccs))
print(f"[bridge] median top-10 Jaccard between {CFG.bridge_steps}-step and "
      f"{CFG.num_steps}-step pilots: {_med:.2f}")
if _med >= 0.6:
    print("[bridge] high overlap: channel identity is consistent across "
          "the two step settings; the 20-step main run connects directly "
          "to the earlier 10-step analyses.")
else:
    print("[bridge][WARN] low overlap: channel identity differs between "
          "step settings. Consider rerunning the experiment at 10 steps "
          "before interpreting cross-paper comparisons.")
DECISION["evidence"]["bridge_10step"] = {
    "bridge_steps": int(CFG.bridge_steps), "t_bridge": int(_tb),
    "per_pilot_jaccard": [float(j) for j in _jaccs],
    "median_jaccard": _med}
save_json_atomic(DECISION, _dec_path)
print("[bridge] recorded in pilot_decision.json")


## S6. Phase B, main capture (100 scenarios, resumable)

For each scenario this saves the five statistics at the analysis block and
the secondary blocks listed in S1, per-token L2 norms at the analysis block,
the full fp16 tensor at the analysis block and step, the generated image and
the top-20 channels by the chosen primary statistic. It also adds a row to
the run log with the agreement between the two ranking statistics. Errors
are logged and the scenario is skipped. Incomplete scenarios are redone
automatically the next time the cell runs.


In [ ]:
# =====================================================================
# S6. Phase B, main capture
# =====================================================================
# Needs pilot_decision.json. Each scenario is a 20-step generation that
# saves stats at l* and the secondary blocks, token norms at l*, the full
# fp16 tensor at (l*, t*), the top-20 by the primary statistic and one
# run_log row. A disconnect only loses the scenario in progress, so just
# run the cell again to resume.
_dec_path = os.path.join(PATHS["meta"], "pilot_decision.json")
if not os.path.exists(_dec_path):
    raise RuntimeError("pilot_decision.json missing -- run Phase A (S4) and the "
                       "decision cell (S5) before Phase B.")
with open(_dec_path) as _f:
    DECISION = json.load(_f)
L_STAR = int(DECISION["l_star"]); T_STAR = int(DECISION["t_star"])
PRIMARY_STAT = DECISION["primary_stat"]
if DECISION["config_hash"] != CONFIG_HASH:
    print(f"[main][WARN] decision was made under config {DECISION['config_hash']}, "
          f"current config is {CONFIG_HASH} -- verify nothing relevant changed.")
STAT_BLOCKS = sorted(set(list(CFG.secondary_blocks) + [L_STAR]))

print("=" * 70)
print(f"PHASE B - MAIN RUN: {len(SCENARIOS)} scenarios | l*={L_STAR} t*={T_STAR} "
      f"primary={PRIMARY_STAT} | stat blocks {STAT_BLOCKS}")
print(f"per scenario: stats {len(STAT_BLOCKS)}x{CFG.num_steps}x5x{CFG.d_model} + "
      f"token norms + fp16 tensor ({N_IMG}x{CFG.d_model} ~24 MB) + png + top-20")
print("=" * 70)

from tqdm.auto import tqdm
_n_ok = _n_skip = _n_warn = _n_err = 0
_pbar = tqdm(SCENARIOS, desc="Phase B")
for _sid, _pid, _seed, _text in _pbar:
    try:
        _diag = run_scenario(
            pipe, CFG, _sid, _pid, _text, _seed, EMBEDS, PATHS["main"],
            stat_blocks=STAT_BLOCKS, n_steps=CFG.num_steps, n_img=N_IMG,
            pipe_dtype=DTYPE, device=DEVICE, lstar=L_STAR, tstar=T_STAR,
            primary_stat=PRIMARY_STAT, capture_full=CFG.save_full_tensor,
            capture_token_norms=True, tag="main",
            extra_meta={"precision_mode": PRECISION_MODE,
                        "config_hash": CONFIG_HASH})
    except Exception as _e:
        _n_err += 1
        append_error_log(ERROR_LOG_PATH, _sid, f"{type(_e).__name__}: {_e}")
        print(f"[main][ERROR] {_sid}: {type(_e).__name__}: {_e} -- continuing")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue
    if _diag["status"] == "skipped":
        _n_skip += 1
    else:
        append_run_log(RUN_LOG_PATH, _diag)
        if _diag["status"] == "ok":
            _n_ok += 1
        else:
            _n_warn += 1
            append_error_log(ERROR_LOG_PATH, _sid, f"warn diag: {_diag}")
        _pbar.set_postfix_str(
            f"top1=ch{_diag.get('top1_channel', '?')} "
            f"J10={_diag.get('jacc_primary_vs_meanabs_top10', float('nan')):.2f} "
            f"{_diag.get('wall_s', 0):.0f}s")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print(f"[main] done: {_n_ok} ok | {_n_skip} skipped(resume) | "
      f"{_n_warn} warn | {_n_err} error")
if _n_warn or _n_err:
    print(f"[main] see {ERROR_LOG_PATH}; warned/errored scenarios that saved "
          f"incompletely will be redone automatically on the next run of this cell.")


## S7. Live diagnostics (run anytime)

Quick checks while the run is going, not final figures. They show
the top-1 channel of each scenario (to spot drift), the agreement between the
two ranking statistics, wall time, the concentration ratio and a contact
sheet of the latest images.


In [ ]:
# =====================================================================
# S7. Live diagnostics
# =====================================================================
# Can run at any time during or after Phase B. Reads meta/run_log.csv and
# the newest main-run images. The output is for checking, not a final figure.

import pandas as pd
from PIL import Image

if not os.path.exists(RUN_LOG_PATH):
    print("[quicklook] no run_log.csv yet")
else:
    _log = pd.read_csv(RUN_LOG_PATH)
    _m = _log[_log["tag"] == "main"].copy()
    if len(_m) == 0:
        print("[quicklook] no main-run rows yet")
    else:
        for c in ("top1_channel", "jacc_primary_vs_meanabs_top10", "wall_s",
                  "top1_over_median", "top1_sign_consistency"):
            _m[c] = pd.to_numeric(_m[c], errors="coerce")
        fig, ax = plt.subplots(2, 2, figsize=(FIG_PAGE, 4.2))
        ax[0, 0].scatter(range(len(_m)), _m["top1_channel"], s=8,
                         color=OKABE_ITO[0])
        ax[0, 0].set_title("top-1 channel per scenario", fontsize=8)
        ax[0, 0].set_xlabel("scenario"); ax[0, 0].set_ylabel("channel index")
        ax[0, 1].hist(_m["jacc_primary_vs_meanabs_top10"].dropna(),
                      bins=np.linspace(0, 1, 21), color=OKABE_ITO[1])
        ax[0, 1].set_title("top-10 agreement of the two statistics", fontsize=8)
        ax[0, 1].set_xlabel("Jaccard index"); ax[0, 1].set_ylabel("scenarios")
        ax[1, 0].plot(_m["wall_s"].values, lw=0.9, color=OKABE_ITO[2])
        ax[1, 0].set_title("wall time per scenario", fontsize=8)
        ax[1, 0].set_xlabel("scenario"); ax[1, 0].set_ylabel("seconds")
        ax[1, 1].plot(_m["top1_over_median"].values, lw=0.9,
                      color=OKABE_ITO[3])
        ax[1, 1].set_yscale("log")
        ax[1, 1].set_title("top-1 over median score", fontsize=8)
        ax[1, 1].set_xlabel("scenario"); ax[1, 1].set_ylabel("ratio")
        save_fig(fig, fig_stem("S7_quicklook"))
        print("[quicklook] completed main rows:", len(_m))
        print("[quicklook] top-1 channel counts:")
        print(_m["top1_channel"].value_counts().head(8).to_string())
        print(f"[quicklook] agreement J10: median="
              f"{_m['jacc_primary_vs_meanabs_top10'].median():.3f}  "
              f"min={_m['jacc_primary_vs_meanabs_top10'].min():.3f}")

_pngs = sorted(glob.glob(os.path.join(PATHS["main"], "*.png")),
               key=os.path.getmtime)[-12:]
if _pngs:
    fig, axes = plt.subplots(3, 4, figsize=(FIG_PAGE, 5.4))
    for _a, _p in zip(axes.flat, _pngs):
        _im = Image.open(_p); _im.thumbnail((256, 256))
        _a.imshow(_im); _a.set_title(os.path.basename(_p)[:-4], fontsize=6)
    for _a in axes.flat:
        _a.axis("off")
    save_fig(fig, fig_stem("S7_contact_sheet"))
else:
    print("[quicklook] no main-run images yet")


## S8. Integrity audit (required before analysis)

Checks every scenario again from disk (files, npz keys, hook fire counts,
NaN flags, tensor shapes) and builds the list of clean scenarios. All
analysis cells use exactly this list and report how many scenarios they
used.


In [ ]:
# =====================================================================
# S8. Post-run integrity audit
# =====================================================================
# Run before any analysis cell. For each scenario it checks that the files
# exist, every hook fired num_steps times, there are no NaNs or hook errors,
# the npz has the needed keys and the tensor has the right shape and dtype.
# Problem scenarios are printed. The analysis cells use whatever passes and
# report n_used.

import pandas as pd

with open(os.path.join(PATHS["meta"], "pilot_decision.json")) as _f:
    DECISION = json.load(_f)
L_STAR = int(DECISION["l_star"]); T_STAR = int(DECISION["t_star"])
PRIMARY_STAT = DECISION["primary_stat"]

_need_tensor = CFG.save_full_tensor
_rows = []
for _sid, _pid, _seed, _ in SCENARIOS:
    _p = scenario_paths(PATHS["main"], _sid, _need_tensor, L_STAR, T_STAR)
    _r = {"sid": _sid, "npz": os.path.exists(_p["npz"]),
          "png": os.path.exists(_p["png"]),
          "tensor": (os.path.exists(_p["tensor"]) if _need_tensor else True),
          "fires_ok": None, "nan_flag": None, "hook_errors": None,
          "keys_ok": None, "tensor_ok": None, "decision_ok": None}
    if _r["npz"]:
        try:
            with np.load(_p["npz"], allow_pickle=False) as _z:
                _meta = meta_from_npz(_z)
                _r["fires_ok"] = bool(_meta["fires_ok"])
                _r["nan_flag"] = bool(_meta["nan_flag"])
                _r["hook_errors"] = int(_meta["hook_errors"])
                _r["decision_ok"] = (_meta.get("lstar") == L_STAR
                                     and _meta.get("tstar") == T_STAR
                                     and _meta.get("primary_stat") == PRIMARY_STAT)
                _need_keys = (["meta_json", "sigmas", "timesteps", "top20_ids",
                               "top20_scores", "token_norms_lstar"] +
                              [f"b{L_STAR:02d}__{s}" for s in STAT_NAMES])
                _r["keys_ok"] = all(k in _z.files for k in _need_keys)
        except Exception as _e:
            _r["keys_ok"] = False
            _r["npz"] = f"unreadable:{type(_e).__name__}"
    if _need_tensor and _r["tensor"] is True:
        try:
            _X = np.load(_p["tensor"], mmap_mode="r")
            _r["tensor_ok"] = (_X.shape == (N_IMG, CFG.d_model)
                               and _X.dtype == np.float16)
        except Exception:
            _r["tensor_ok"] = False
    elif not _need_tensor:
        _r["tensor_ok"] = True
    _rows.append(_r)

audit_df = pd.DataFrame(_rows)
audit_df["clean"] = (
    (audit_df["npz"] == True) & (audit_df["png"] == True)
    & (audit_df["tensor"] == True) & (audit_df["fires_ok"] == True)
    & (audit_df["nan_flag"] == False) & (audit_df["hook_errors"] == 0)
    & (audit_df["keys_ok"] == True) & (audit_df["tensor_ok"] == True)
    & (audit_df["decision_ok"] == True))
audit_df.to_csv(os.path.join(PATHS["meta"], "audit.csv"), index=False)

_n_clean = int(audit_df["clean"].sum())
print(f"[audit] {_n_clean}/{len(audit_df)} scenarios fully clean")
if _n_clean < len(audit_df):
    print("[audit] PROBLEM SCENARIOS (delete their partial files or just re-run "
          "the Phase-B cell -- incomplete scenarios are redone automatically):")
    print(audit_df[~audit_df["clean"]].to_string(index=False))
else:
    print("[audit] AUDIT PASSED -- proceed to deliverables D1-D4.")
CLEAN_SCENARIOS = [t for t, ok in zip(SCENARIOS, audit_df["clean"].tolist()) if ok]
print(f"[audit] CLEAN_SCENARIOS: {len(CLEAN_SCENARIOS)} "
      f"(analysis cells use exactly this set)")


## S9. D1, activation landscape

The scenario mean of the absolute hidden state at the analysis block and
step, streamed from the saved tensors and summed in float64. Channels are
colored by category relative to the median channel. A channel counts as high
at 10 times the median statistic and as massive at 100 times.

There are four figures. The first three are full-width and show the
per-channel profile, the token by channel heatmap and the signed landscape
on a diverging scale that is dark at zero. They label the five largest
channels and up to ten recurring channels. The fourth is a 3D bar plot. It marks up to six massive
channels as tick labels on the channel axis and leaves out ids that sit too
close to each other or to the ends of the axis.


In [ ]:
# =====================================================================
# LIB E. Analysis functions
# =====================================================================
# numpy and pandas only. The main run saves the full tensors
# (CFG.save_full_tensor), so every analysis can be rebuilt from disk. The
# D1 landscape streams the per-scenario fp16 tensors and sums in float64.
import os, json
import numpy as np
import pandas as pd

def load_decision(meta_dir):
    with open(os.path.join(meta_dir, "pilot_decision.json")) as f:
        return json.load(f)

def stream_landscape_and_uniformity(tensor_paths, n_img, d):
    """One pass over the saved fp16 tensors, summed in float64. Returns

      mean_abs_landscape     [n_img, d]  scenario mean of |X| (D1)
      mean_signed_landscape  [n_img, d]  scenario mean of X, keeps the sign
      frac_wall_mean         [d]  mean fraction of tokens with |X| > 0.5 * max,
                                  high for wall-like channels
      cv_abs_mean            [d]  mean coefficient of variation of |X| over tokens
    """
    sum_abs = np.zeros((n_img, d), np.float64)
    sum_signed = np.zeros((n_img, d), np.float64)
    frac_wall_sum = np.zeros(d, np.float64)
    cv_sum = np.zeros(d, np.float64)
    n_used = 0
    for p in tensor_paths:
        X = np.load(p).astype(np.float32)
        if X.shape != (n_img, d):
            raise ValueError(f"{p}: shape {X.shape}, expected {(n_img, d)}")
        A = np.abs(X.astype(np.float64))
        sum_abs += A
        sum_signed += X
        colmax = np.maximum(A.max(axis=0), 1e-12)
        frac_wall_sum += (A > 0.5 * colmax[None, :]).mean(axis=0)
        mu = A.mean(axis=0)
        cv_sum += A.std(axis=0) / np.maximum(mu, 1e-12)
        n_used += 1
    n = max(n_used, 1)
    return {"n_used": n_used,
            "mean_abs_landscape": (sum_abs / n).astype(np.float32),
            "mean_signed_landscape": (sum_signed / n).astype(np.float32),
            "frac_wall_mean": frac_wall_sum / n,
            "cv_abs_mean": cv_sum / n}

def collect_top_lists(main_dir, scenarios, save_tensor, lstar, tstar):
    """Returns top_ids [S, top_save], top_scores [S, top_save] and sids."""
    ids, scores, sids = [], [], []
    for sid, pid, seed, _ in scenarios:
        p = scenario_paths(main_dir, sid, save_tensor, lstar, tstar)
        with np.load(p["npz"], allow_pickle=False) as z:
            ids.append(z["top20_ids"].astype(np.int64))
            scores.append(z["top20_scores"].astype(np.float64))
        sids.append(sid)
    return np.stack(ids), np.stack(scores), sids

def recurrence_freqs(top_ids, k_list, d):
    """f_k(ch) is the fraction of scenarios that have ch in their top-k by
    the primary statistic. Returns one entry per k in k_list plus one for
    the full list width (top-20)."""
    S, width = top_ids.shape
    out = {}
    for k in sorted(set(list(k_list) + [width])):
        c = np.zeros(d, np.float64)
        for row in top_ids:
            c[row[:k]] += 1.0
        out[int(k)] = c / S
    return out

def coverage_stats(top10_counts):
    """counts[d] is the number of scenarios with d in their top-10. Returns
    the counts sorted in descending order, their cumulative fraction and the
    number of channels that cover 95 percent of all memberships."""
    c = np.sort(np.asarray(top10_counts, np.float64))[::-1]
    total = max(c.sum(), 1e-12)
    cum = np.cumsum(c) / total
    n95 = int(np.searchsorted(cum, 0.95) + 1)
    return c, cum, n95

def scenario_score_vector(npz_path, block, tstar, primary):
    with np.load(npz_path, allow_pickle=False) as z:
        stats_t = {s: z[f"b{block:02d}__{s}"][tstar] for s in STAT_NAMES}
    vec = primary_score(stats_t, primary)
    extra = {s: np.asarray(stats_t[s], np.float64)
             for s in ("frac_pos", "p99_abs", "max_abs")}
    return vec, extra

def per_step_scores(npz_path, block, primary, channels):
    """Primary score per step for the given channels, [n_steps, len(channels)]."""
    ch = np.asarray(channels, np.int64)
    with np.load(npz_path, allow_pickle=False) as z:
        if primary == "abs_signed_mean":
            arr = np.abs(z[f"b{block:02d}__signed_mean"][:, ch])
        elif primary == "mean_abs":
            arr = z[f"b{block:02d}__mean_abs"][:, ch]
        else:
            raise ValueError(primary)
    return arr.astype(np.float64)

def build_candidate_table(scenarios, main_dir, lstar, tstar, primary, freqs,
                          uniformity, anchors, save_tensor, d):
    """D3 table with one row per channel in the union of per-scenario top-20 sets."""
    U = np.where(freqs[max(freqs)] > 0)[0]         # union of top-20 members
    S = len(scenarios)
    per_scores = np.zeros((S, len(U)))
    per_ranks = np.zeros((S, len(U)), np.int64)
    cons_sum = np.zeros(len(U))
    p99max_sum = np.zeros(len(U))
    for i, (sid, pid, seed, _) in enumerate(scenarios):
        p = scenario_paths(main_dir, sid, save_tensor, lstar, tstar)
        vec, extra = scenario_score_vector(p["npz"], lstar, tstar, primary)
        order = np.argsort(-vec, kind="stable")
        rank_full = np.empty(d, np.int64)
        rank_full[order] = np.arange(1, d + 1)
        per_scores[i] = vec[U]
        per_ranks[i] = rank_full[U]
        cons_sum += sign_consistency(extra["frac_pos"])[U]
        p99max_sum += extra["p99_abs"][U] / np.maximum(extra["max_abs"][U], 1e-12)
    sm, ss = per_scores.mean(axis=0), per_scores.std(axis=0)
    df = pd.DataFrame({
        "channel": U,
        "f1": freqs[1][U], "f5": freqs[5][U], "f10": freqs[10][U],
        "f20": freqs[max(freqs)][U],
        "score_mean": sm, "score_std": ss,
        "score_cv": ss / np.maximum(sm, 1e-12),
        "rank_mean": per_ranks.mean(axis=0),
        "rank_worst": per_ranks.max(axis=0),
        "sign_consistency_mean": cons_sum / S,
        "p99_over_max_mean": p99max_sum / S,
        "frac_wall": np.asarray(uniformity["frac_wall_mean"])[U],
        "cv_abs_tokens": np.asarray(uniformity["cv_abs_mean"])[U],
        "is_dg_anchor": [int(ch) in set(int(a) for a in anchors) for ch in U],
    })
    return df.sort_values(["f10", "f5", "score_mean"],
                          ascending=False).reset_index(drop=True)

def variance_decomposition(mat):
    """One-way variance decomposition over prompts for mat [n_prompts, n_seeds].
    eta2 near 1 means the magnitude is driven by the prompt. eta2 near 0
    means it doesn't depend on the prompt."""

    mat = np.asarray(mat, np.float64)
    grand = mat.mean()
    pm = mat.mean(axis=1)
    ssb = mat.shape[1] * ((pm - grand) ** 2).sum()
    ssw = ((mat - pm[:, None]) ** 2).sum()
    return {"eta2_prompt": float(ssb / max(ssb + ssw, 1e-24)),
            "ss_between": float(ssb), "ss_within": float(ssw)}

print("[lib-analysis] loaded (D1-D4 functions)")


In [ ]:
# =====================================================================
# S9. D1, token x channel activation landscape
# =====================================================================
# Scenario mean of |X[token, channel]| at (l*, t*), streamed from the
# saved fp16 tensors and summed in float64. Four figures follow.
#   a  per-channel profile, colored by category
#   b  token x channel heatmap on a log color scale
#   c  signed landscape on a diverging colormap that is dark at zero
#   d  3D bar landscape with the massive channel ids on the axis
assert "CLEAN_SCENARIOS" in globals(), "run the S8 audit cell first"
DECISION = load_decision(PATHS["meta"])
L_STAR, T_STAR = int(DECISION["l_star"]), int(DECISION["t_star"])
PRIMARY_STAT = DECISION["primary_stat"]

_tpaths = [scenario_paths(PATHS["main"], sid, True, L_STAR, T_STAR)["tensor"]
           for sid, _, _, _ in CLEAN_SCENARIOS]
_gib = len(_tpaths) * N_IMG * CFG.d_model * 2 / 2**30
print(f"[D1] streaming {len(_tpaths)} tensors ({_gib:.1f} GiB total read) ...")
_t0 = time.time()
LAND = stream_landscape_and_uniformity(_tpaths, N_IMG, CFG.d_model)
print(f"[D1] landscape built from {LAND['n_used']} scenarios "
      f"in {time.time()-_t0:.0f}s")
save_npz_atomic(
    os.path.join(PATHS["analysis"], "landscape.npz"),
    mean_abs_landscape=LAND["mean_abs_landscape"],
    mean_signed_landscape=LAND["mean_signed_landscape"],
    frac_wall_mean=LAND["frac_wall_mean"].astype(np.float32),
    cv_abs_mean=LAND["cv_abs_mean"].astype(np.float32),
    meta_json=meta_to_npz_field({
        "n_used": LAND["n_used"], "l_star": L_STAR, "t_star": T_STAR,
        "config_hash": CONFIG_HASH, "precision_mode": PRECISION_MODE}))

L = LAND["mean_abs_landscape"]
prof = L.mean(axis=0)
top5 = topk_ids(prof, 5)
CATS, _med = classify_channels(prof)
_massive_ids = [int(c) for c in np.where(CATS == 2)[0]]
_high_ids = [int(c) for c in np.where(CATS == 1)[0]]
print(f"[D1] top-5 channels by scenario-mean mean|x|: "
      f"{[(int(c), round(float(prof[c]), 2)) for c in top5]}")
print(f"[D1] categorization vs median ({_med:.2f}): "
      f"{len(_massive_ids)} massive (>= {MASSIVE_MULT:g}x): {_massive_ids} | "
      f"{len(_high_ids)} high (>= {HIGH_MULT:g}x)")

# Recurring channels from the per-scenario top-20 lists. Every channel that
# D2 reports should be easy to find in these figures, however small it is.
_ti_d1, _, _ = collect_top_lists(PATHS["main"], CLEAN_SCENARIOS,
                                 CFG.save_full_tensor, L_STAR, T_STAR)
FREQS = recurrence_freqs(_ti_d1, list(CFG.k_list), CFG.d_model)
RECURRING = [int(c) for c in np.where(
    FREQS[10] >= CFG.recurring_set_threshold)[0]]
print(f"[D1] recurring channels (f10 >= {CFG.recurring_set_threshold}): "
      f"{RECURRING}")
_label_ids = sorted(set([int(c) for c in top5] + RECURRING[:10]))

_scen_txt = (f"averaged over {LAND['n_used']} scenarios from the "
             f"{CFG.n_prompts} prompt by {len(CFG.seeds)} seed grid")

# ---- (a) per-channel profile, colored by category
fig, ax = plt.subplots(figsize=(FIG_PAGE, 2.4))
category_profile(ax, prof, CATS)
annotate_points(ax, _label_ids, [float(prof[c]) for c in _label_ids],
                _label_ids)
ax.set_xlabel("channel index")
ax.set_ylabel(r"token mean of mean $|x|$")
legend_above(ax, ncol=3)
fig_title(ax, f"{MODEL_LABEL}: per-channel mean $|x|$ at layer {L_STAR}, "
              f"step {T_STAR} ({LAND['n_used']} scenarios)", pad=22)
save_fig(fig, fig_stem("D1_channel_profile"), caption=(
    f"Per-channel profile of the activation landscape of {MODEL_LABEL} at "
    f"block {L_STAR}, denoising step {T_STAR}: the mean over the {N_IMG} "
    f"image tokens of the scenario-mean absolute activation, {_scen_txt}. "
    f"{CAT_CAPTION} Labels name the largest channels and every globally "
    f"recurring channel reported by the recurrence analysis; labels of "
    f"adjacent channels are placed to the left and right of their "
    f"points."))

# ---- (b) token x channel heatmap
_pool = max(1, CFG.d_model // 1024)
_TPOOL = 8            # max-pool tokens so sparse bright cells survive
_Dp = (CFG.d_model // _pool) * _pool
_Lp = L[:, :_Dp].reshape(N_IMG, _Dp // _pool, _pool).max(axis=2)
_Lp = _Lp.reshape(N_IMG // _TPOOL, _TPOOL, -1).max(axis=1)
_vmin = max(float(np.percentile(_Lp[_Lp > 0], HEAT_FLOOR_PCTL)), 1e-6)
# Clip the color scale at the 99.9th percentile. The massive channels then
# saturate (they are labeled anyway) and the other elevated channels move
# up the ramp where they can be seen.
_vmax = max(float(np.percentile(_Lp, 99.9)), _vmin * 10.0)
fig, ax = plt.subplots(figsize=(FIG_PAGE, 3.0))
im = ax.imshow(_Lp, aspect="auto", origin="lower", interpolation="nearest",
               cmap="magma", norm=LogNorm(vmin=_vmin, vmax=_vmax),
               extent=[0, _Dp, 0, N_IMG], rasterized=True)
ax.set_xlabel("channel index"); ax.set_ylabel("image token index")
label_channels_top(ax, _label_ids, _label_ids)
fig_title(ax, f"{MODEL_LABEL}: mean $|x|$ per channel and image token "
              f"(layer {L_STAR}, step {T_STAR}, {LAND['n_used']} "
              f"scenarios)", pad=22)
cb = fig.colorbar(im, ax=ax, pad=0.01, extend="max")
cb.set_label(r"mean $|x|$ (log scale)")
save_fig(fig, fig_stem("D1_landscape"), caption=(
    f"Token by channel activation landscape of {MODEL_LABEL} at block "
    f"{L_STAR}, step {T_STAR}: the scenario-mean absolute activation, "
    f"{_scen_txt}. The largest channels and every globally recurring "
    f"channel are marked above the panel with their channel indices."
    + f" For rendering, tokens are max-pooled in groups of {_TPOOL}"
    + (f" and the channel axis by a factor of {_pool}" if _pool > 1 else "")
    + (", so spatially sparse elevated cells survive the raster; the "
       "color scale is clipped at the 99.9th percentile, so the massive "
       "channels saturate the brightest color while other elevated "
       "channels remain distinguishable. Tokens are indexed row-major "
       "over the 64 by 64 latent grid.")))

# ---- (c) signed landscape
# Pool by keeping the signed value of largest magnitude, so isolated
# massive columns keep their sign and still show up in print.
Ls = LAND["mean_signed_landscape"]
_Lsr = Ls[:, :_Dp].reshape(N_IMG, _Dp // _pool, _pool)
_ami = np.abs(_Lsr).argmax(axis=2)
_Lsp = np.take_along_axis(_Lsr, _ami[..., None], axis=2)[..., 0]
_Lst = _Lsp.reshape(N_IMG // _TPOOL, _TPOOL, -1)
_ami2 = np.abs(_Lst).argmax(axis=1)
_Lsp = np.take_along_axis(_Lst, _ami2[:, None, :], axis=1)[:, 0, :]
_lin = max(float(np.median(np.abs(_Lsp))), 1e-6)
_vmax = max(float(np.percentile(np.abs(_Lsp), 99.9)), _lin * 10.0)
fig, ax = plt.subplots(figsize=(FIG_PAGE, 3.0))
im = ax.imshow(_Lsp, aspect="auto", origin="lower", interpolation="nearest",
               cmap=SIGNED_CMAP,
               norm=SymLogNorm(linthresh=_lin, linscale=0.5,
                               vmin=-_vmax, vmax=_vmax),
               extent=[0, _Dp, 0, N_IMG], rasterized=True)
ax.set_xlabel("channel index"); ax.set_ylabel("image token index")
label_channels_top(ax, _label_ids, _label_ids)
fig_title(ax, f"{MODEL_LABEL}: signed mean activation per channel and "
              f"image token (layer {L_STAR}, step {T_STAR})", pad=22)
cb = fig.colorbar(im, ax=ax, pad=0.01, extend="both")
cb.set_label(r"mean $x$ (signed, symmetric log scale)")
save_fig(fig, fig_stem("D1_landscape_signed"), caption=(
    f"Signed counterpart of the activation landscape of {MODEL_LABEL} at "
    f"block {L_STAR}, step {T_STAR}, on a symmetric log color scale "
    f"centered at zero: positive values follow the same black-to-yellow "
    f"scheme as the magnitude landscape, negative values its blue "
    f"counterpart, and near-zero values stay black. A channel whose "
    f"column mixes warm and cool colors has inconsistent sign across "
    f"tokens, which the absolute signed mean statistic would partially "
    f"cancel. Values within the median magnitude stay black. The "
    f"largest channels and every globally recurring channel are marked "
    f"above the panel."
    + f" For rendering, tokens are pooled in groups of {_TPOOL}"
    + (f" and channels by a factor of {_pool}" if _pool > 1 else "")
    + (", keeping the signed value of largest magnitude in each group, "
       "and the color scale is clipped at the 99.9th percentile of "
       "magnitudes.")))

# ---- (d) 3D bar landscape
_TOKB, _CHB = 32, 96
assert N_IMG % _TOKB == 0 and CFG.d_model % _CHB == 0
_tw, _cw = N_IMG // _TOKB, CFG.d_model // _CHB
_Z = L.reshape(_TOKB, _tw, _CHB, _cw).max(axis=(1, 3))     # [TOKB, CHB]
_binc = CATS.reshape(_CHB, _cw).max(axis=1)                 # bin category
_xs, _ys = np.meshgrid(np.arange(_TOKB) * _tw,
                       np.arange(_CHB) * _cw, indexing="ij")
_colors = np.array([BAR3D_COLORS[int(c)] for c in _binc])[None, :] \
    .repeat(_TOKB, axis=0)
# layout="none" because constrained layout drops 3D axis labels. The fixed
# axes position leaves room for the three axis labels, which drift a little
# with the raster dpi, so they stay on the canvas at any resolution.
fig = plt.figure(figsize=(FIG_PAGE * 0.68, 3.9), layout="none")
ax = fig.add_subplot(111, projection="3d")
ax.set_position((0.0, 0.10, 0.82, 0.90))
ax.bar3d(_xs.ravel(), _ys.ravel(), np.zeros(_Z.size),
         _tw * 0.9, _cw * 0.8, _Z.ravel(),
         color=list(_colors.ravel()),
         shade=True, edgecolor="none", zsort="max")
for _pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
    _pane.set_facecolor("#F3F5FA"); _pane.set_alpha(1.0)
# Axis labels are drawn as figure text. mplot3d places its own labels
# differently across matplotlib versions and resolutions and sometimes clips
# or drops them. Figure text at fixed coordinates looks the same everywhere.
ax.set_xlabel(""); ax.set_ylabel(""); ax.set_zlabel("")
# The title is figure text too, since the layout engine is off. It sits at
# the top of the canvas above the 3D axes, where the cube never reaches, so
# it can't overlap bars, panes or tick labels.
fig.text(0.41, 0.975, f"{MODEL_LABEL}: activation landscape at layer "
         f"{L_STAR} (linear scale)", ha="center", va="top",
         fontsize=_BASE_PT + 0.5)
fig.text(0.28, 0.075, "image token index", ha="center",
         fontsize=_BASE_PT, rotation=-13)
fig.text(0.795, 0.145, "channel index", ha="center",
         fontsize=_BASE_PT, rotation=42)
fig.text(0.945, 0.55, r"mean $|x|$ (linear scale)", va="center",
         ha="center", rotation=90, fontsize=_BASE_PT)
ax.tick_params(pad=1, labelsize=_BASE_PT - 2)
# Massive channel ids become ticks on the channel axis instead of floating
# text. Ids too close to each other or to the axis ends are dropped so the
# tick labels don't overlap.

_tick_ids = []
for _c in _massive_ids[:6]:
    if (min(_c, CFG.d_model - _c) > 0.05 * CFG.d_model
            and all(abs(_c - _k) > 0.05 * CFG.d_model for _k in _tick_ids)):
        _tick_ids.append(_c)
_yticks = sorted(set([0, CFG.d_model] + _tick_ids))
ax.set_yticks(_yticks)
ax.set_yticklabels([str(t) for t in _yticks])
for _tl, _tv in zip(ax.get_yticklabels(), _yticks):
    if _tv in _tick_ids:
        _tl.set_color(BAR3D_COLORS[2])
        _tl.set_fontweight("bold")
ax.view_init(elev=28, azim=-58)
save_fig(fig, fig_stem("D1_landscape_3d"), caption=(
    f"Three-dimensional bar rendering of the activation landscape of "
    f"{MODEL_LABEL} at block {L_STAR}, step {T_STAR}: the scenario mean "
    f"of the absolute activation, {_scen_txt}. The vertical axis is "
    f"linear, following the rendering of Gan et al.; on a linear scale "
    f"typical channels form a flat floor and massive channels stand out "
    f"as walls. For rendering, tokens are aggregated into {_TOKB} bins "
    f"and channels into {_CHB} bins by the maximum, which preserves "
    f"isolated massive channels. Bars are blue for typical channels, "
    f"light purple for high channels (at least {HIGH_MULT:g} times the "
    f"median channel), and deep purple for massive channels (at least "
    f"{MASSIVE_MULT:g} times the median); massive channels appear as "
    f"their own bold tick labels on the channel axis."), tight=False)
print("[D1] figures saved to", PATHS["figures"])


## S9b. D1 per layer

Per-layer views built from the pilot, which logs the per-channel statistics
at every layer. Here a layer means a transformer block, indexed the same way
as everywhere else. These views are per channel and averaged over tokens.
The token-resolved landscape only exists at the analysis layer, where the
main run saves full tensors.

The cell draws a layer by channel heatmap over all layers, where massive
channels show up as bright vertical lines. It also draws the channel profile
averaged over the high-norm band, one small profile per band layer plus a
control layer on each side, and one profile file per layer under
figures/per_layer.


In [ ]:
# =====================================================================
# S9b. D1 per layer, from the pilot statistics at every block
# =====================================================================
# The pilot logs the five per-channel statistics at every block, so these
# views are per channel and aggregated over tokens. Token-resolved maps
# only exist at l* (D1 above). Everything is the median over the pilot
# scenarios at the analysis step t*. Outputs are a layer x channel
# heatmap, the band-averaged profile, small profiles for the band layers
# and one profile file per block under figures/per_layer/.
DECISION = load_decision(PATHS["meta"])
L_STAR, T_STAR = int(DECISION["l_star"]), int(DECISION["t_star"])
_pilot_sids = [f"pilot__{scenario_id(p, s)}" for p, s in CFG.pilot_pairs]
for _sid in _pilot_sids:
    assert os.path.exists(os.path.join(PATHS["pilot"], f"{_sid}.npz")), \
        f"pilot file missing: {_sid} (run S4 first)"

_nb = CFG.n_blocks_expected
_zs = {s: np.load(os.path.join(PATHS["pilot"], f"{s}.npz"),
                  allow_pickle=False) for s in _pilot_sids}
PROF_LAYERS = np.zeros((_nb, CFG.d_model), np.float64)
for _b in range(_nb):
    PROF_LAYERS[_b] = np.median(np.stack(
        [_z[f"b{_b:02d}__mean_abs"][T_STAR].astype(np.float64)
         for _z in _zs.values()]), axis=0)
for _z in _zs.values():
    _z.close()

_top5_l = topk_ids(PROF_LAYERS[L_STAR], 5)
_lo, _hi = int(CFG.band[0]), int(CFG.band[1])
# recurring channels, only available once the main run and audit exist
_rec_l = []
if "CLEAN_SCENARIOS" in globals() and "collect_top_lists" in globals():
    try:
        _ti9, _, _ = collect_top_lists(PATHS["main"], CLEAN_SCENARIOS,
                                       CFG.save_full_tensor, L_STAR, T_STAR)
        _fr9 = recurrence_freqs(_ti9, list(CFG.k_list), CFG.d_model)
        _rec_l = [int(c) for c in np.where(
            _fr9[10] >= CFG.recurring_set_threshold)[0]][:10]
    except Exception as _e:
        print(f"[D1-layers][note] recurring set unavailable ({_e})")
_label_ids_l = sorted(set([int(c) for c in _top5_l] + _rec_l))
print(f"[D1-layers] per-layer profiles built ({_nb} blocks x "
      f"{CFG.d_model} channels, median of {len(_pilot_sids)} pilots, "
      f"step {T_STAR})")

# ---- (a) layer x channel heatmap (channel index on the bottom axis)
_pos = PROF_LAYERS[PROF_LAYERS > 0]
_vmin = max(float(np.percentile(_pos, HEAT_FLOOR_PCTL)), 1e-6)
_vmax = max(float(np.percentile(PROF_LAYERS, 99.9)), _vmin * 10.0)
fig, ax = plt.subplots(figsize=(FIG_PAGE, 2.9))
im = ax.imshow(PROF_LAYERS, aspect="auto", origin="lower",
               interpolation="nearest", cmap="magma",
               norm=LogNorm(vmin=_vmin, vmax=_vmax),
               extent=[0, CFG.d_model, -0.5, _nb - 0.5], rasterized=True)
ax.set_xlabel("channel index")
ax.set_ylabel("layer index")
label_channels_top(ax, _label_ids_l, _label_ids_l)
fig_title(ax, f"{MODEL_LABEL}: per-channel mean $|x|$ across all {_nb} "
              f"layers (step {T_STAR})", pad=22)
cb = fig.colorbar(im, ax=ax, pad=0.01, extend="max")
cb.set_label(r"mean $|x|$ (log scale)")
save_fig(fig, fig_stem("D1_layers_heatmap"), caption=(
    f"Channel magnitudes across layers in {MODEL_LABEL} at denoising "
    f"step {T_STAR}: each row is one layer (transformer block) and each "
    f"column one channel, colored by the per-channel mean absolute "
    f"activation over the image tokens (median over the pilot "
    f"scenarios), on the same black-to-yellow log color scale as the "
    f"token by channel landscape, with the color floor at the "
    f"{HEAT_FLOOR_PCTL:g}th percentile and its top clipped at the 99.9th "
    f"percentile, so no channel is dimmed by the massive channel's peak. A massive channel appears as a bright vertical "
    f"segment spanning exactly the layers in which it exists (layers "
    f"{_lo} to {_hi} form the pre-registered band; the analysis layer is "
    f"{L_STAR}); segments covering a subset of layers therefore reflect "
    f"the model, not a rendering artifact. Labels above the panel mark "
    f"the largest channels at the analysis layer and every globally "
    f"recurring channel."))

# ---- (b) band-aggregate profile
BAND_PROF = PROF_LAYERS[_lo:_hi + 1].mean(axis=0)
_cats_band, _medb = classify_channels(BAND_PROF)
_lab_band = sorted(set([int(c) for c in topk_ids(BAND_PROF, 5)] + _rec_l))
fig, ax = plt.subplots(figsize=(FIG_PAGE, 2.4))
category_profile(ax, BAND_PROF, _cats_band)
annotate_points(ax, _lab_band, [float(BAND_PROF[c]) for c in _lab_band],
                _lab_band)
ax.set_xlabel("channel index")
ax.set_ylabel(r"mean $|x|$, averaged over the band")
legend_above(ax, ncol=3)
fig_title(ax, f"{MODEL_LABEL}: channel profile averaged over the "
              f"high-norm band (layers {_lo} to {_hi})", pad=22)
save_fig(fig, fig_stem("D1_band_aggregate_profile"), caption=(
    f"Band-aggregate channel profile of {MODEL_LABEL}: the per-channel "
    f"mean absolute activation at step {T_STAR}, averaged over layers "
    f"{_lo} to {_hi} (the pre-registered band; median over the pilot "
    f"scenarios). {CAT_CAPTION} Labels name the largest channels and "
    f"every globally recurring channel; labels of adjacent channels are "
    f"placed to the left and right of their points."))
print(f"[D1-layers] band aggregate (layers {_lo} to {_hi}): labeled "
      f"channels = {_lab_band}")

# ---- (c) small multiples over the band, plus a control layer on each side

_panels = sorted(set([max(0, _lo - 2)] + list(range(_lo, _hi + 1))
                     + [min(_nb - 1, _hi + 2)]))
_nc = 6
_nr = int(np.ceil(len(_panels) / _nc))
fig, axes = plt.subplots(_nr, _nc, figsize=(FIG_PAGE, 1.35 * _nr),
                         sharex=True, sharey=True)
for _ax, _b in zip(np.atleast_1d(axes).flat, _panels):
    _v = PROF_LAYERS[_b]
    _c, _ = classify_channels(_v)
    _ax.semilogy(_v, lw=0.25, color="0.75")
    for _k in (1, 2):
        _idx = np.where(_c == _k)[0]
        if len(_idx):
            _ax.scatter(_idx, _v[_idx], s=2 if _k == 1 else 4,
                        color=CAT_COLORS[_k], linewidths=0)
    _in_band = _lo <= _b <= _hi
    _ax.set_title(f"layer {_b}" + ("" if _in_band else "*")
                  + f"  (top {int(np.argmax(_v))})",
                  fontsize=5.5, pad=2,
                  color="0.1" if _in_band else "0.45")
    _ax.tick_params(labelsize=5)
for _ax in np.atleast_1d(axes).flat[len(_panels):]:
    _ax.axis("off")
fig.suptitle(f"{MODEL_LABEL}: per-layer channel profiles across the "
             f"high-norm band (step {T_STAR})", fontsize=_BASE_PT + 0.5)
save_fig(fig, fig_stem("D1_layer_profiles_band"), caption=(
    f"Per-layer channel profiles of {MODEL_LABEL} across the "
    f"pre-registered band (layers {_lo} to {_hi}) with flanking control "
    f"layers outside the band (marked with an asterisk), at step "
    f"{T_STAR} (median over the pilot scenarios, log scale). "
    f"{CAT_CAPTION} Each panel title names its layer and the index of "
    f"its largest channel; titles sit outside the plotting area."))

# ---- (d) one profile file per block (batch, not displayed)
_pl_dir = os.path.join(PATHS["figures"], "per_layer")
os.makedirs(_pl_dir, exist_ok=True)
for _b in range(_nb):
    _v = PROF_LAYERS[_b]
    _c, _ = classify_channels(_v)
    _t3 = topk_ids(_v, 3)
    f2, a2 = plt.subplots(figsize=(FIG_COL, 1.8))
    category_profile(a2, _v, _c, top_ids=_t3)
    a2.set_xlabel("channel index", fontsize=6)
    a2.set_ylabel(r"mean $|x|$", fontsize=6)
    a2.tick_params(labelsize=5.5)
    a2.set_title(f"layer {_b}", fontsize=6, loc="left", pad=2)
    for _ext in ("pdf", "png"):
        f2.savefig(os.path.join(_pl_dir, f"D1_profile_block{_b:02d}.{_ext}"),
                   bbox_inches="tight", dpi=300 if _ext == "png" else None)
    plt.close(f2)
print(f"[D1-layers] wrote {_nb} per-layer profile figures -> {_pl_dir}")


## S9c. Band-layer landscapes (token-resolved)

The main run only stores the token by channel landscape at the analysis
layer. The first cell below captures it at every layer of the high-norm
band, at the analysis step, by running the pilot scenarios once more with
extra hooks. It saves the mean over the pilot scenarios to
analysis/band_landscapes.npz. If a matching capture already exists the cell
skips itself and leaves the file alone.

The second cell draws one full-size landscape per band layer (under
figures/band_landscapes), a grid of all band layers on one shared color
scale and the landscape averaged over the band. The stored fp16 pilot mean
takes at most about 0.07 GiB before compression.


In [ ]:
# =====================================================================
# S9c. Band-layer landscape capture
# =====================================================================
# Writes analysis/band_landscapes.npz with |x| per token and channel at
# every band layer, averaged over the pilot scenarios. Skipped when a
# capture with the same config hash, (l*, t*) and band already exists.
import gc

_dec_path = os.path.join(PATHS["meta"], "pilot_decision.json")
assert os.path.exists(_dec_path), "run the S5 decision cell first"
with open(_dec_path) as _f:
    _DEC9 = json.load(_f)
L_STAR = int(_DEC9["l_star"]); T_STAR = int(_DEC9["t_star"])
_lo9, _hi9 = int(CFG.band[0]), int(CFG.band[1])
_band_blocks = list(range(_lo9, _hi9 + 1))
_bl_path = os.path.join(PATHS["analysis"], "band_landscapes.npz")
_want_meta = {"config_hash": CONFIG_HASH, "lstar": L_STAR,
              "tstar": T_STAR, "band": [_lo9, _hi9]}


def _band_capture_ok(path):
    """True if the npz opens, has the landscape stack and its meta matches
    the current config hash, decision and band."""
    if not os.path.exists(path):
        return False
    try:
        with np.load(path, allow_pickle=False) as z:
            if not ("landscape" in z.files and "blocks" in z.files
                    and "meta_json" in z.files):
                return False
            m = meta_from_npz(z)
            return all(m.get(k) == v for k, v in _want_meta.items())
    except Exception:
        return False


if _band_capture_ok(_bl_path):
    print(f"[S9c] band landscapes already captured -> {_bl_path} (skip)")
else:
    assert "pipe" in globals(), "load the pipeline (S3) before this cell"
    _gb = len(_band_blocks) * N_IMG * CFG.d_model * 2 / 2 ** 30
    print(f"[S9c] capturing |x| landscapes at blocks {_lo9}..{_hi9} "
          f"({len(_band_blocks)} layers), step {T_STAR}, "
          f"{len(CFG.pilot_pairs)} pilot scenarios "
          f"(~{_gb:.2f} GiB stored as the fp16 pilot mean)")

    class BandLandscapeGrabber:
        """Grabs |x| of the image stream (token by channel, conditional
        row) at one denoising step for a set of blocks. The image stream is
        pulled out by an ActivationRecorder, so batching, row selection and
        tuple outputs are handled exactly as in the other captures. That
        recorder is never attached. Only the grabber's own hooks run."""

        def __init__(self, transformer, blocks, tstar):
            _exp = 2 if (CFG.guidance_mode == "cfg_batch2"
                         and CFG.guidance_scale > 1.0) else 1
            self._ex = ActivationRecorder(
                transformer, n_steps=CFG.num_steps, n_img=N_IMG,
                n_txt=int(CFG.n_txt_in_seq), d_model=CFG.d_model,
                stat_blocks=list(blocks),
                expected_batch=_exp, select_row=CFG.cond_row)
            self.blocks = sorted(self._ex.stat_blocks)
            self.tstar = int(tstar)
            self._handles = []
            self.reset()

        def reset(self):
            self.counters = {b: 0 for b in self.blocks}
            self.tensors = {}
            self.errors = []

        def _make_hook(self, bidx):
            def hook(module, args, output):
                step = self.counters[bidx]
                self.counters[bidx] = step + 1
                if step != self.tstar:
                    return
                try:
                    X = self._ex._extract_image_stream(output)
                    if X.shape != (N_IMG, CFG.d_model):
                        raise RuntimeError(f"shape {tuple(X.shape)} != "
                                           f"({N_IMG}, {CFG.d_model})")
                    self.tensors[bidx] = (X.to(torch.float32).abs()
                                          .clamp(max=65504.0)
                                          .to("cpu", torch.float16).numpy())
                except Exception as e:
                    self.errors.append(
                        f"block {bidx} step {step}: {type(e).__name__}: {e}")
            return hook

        def attach(self):
            assert not self._handles
            want = set(self.blocks)
            for i, mod, _k in self._ex.all_blocks:
                if i in want:
                    self._handles.append(
                        mod.register_forward_hook(self._make_hook(i)))
            return self

        def detach(self):
            for h in self._handles:
                h.remove()
            self._handles = []

    _SUM = np.zeros((len(_band_blocks), N_IMG, CFG.d_model), np.float32)
    _n_used9 = 0
    device, pipe_dtype = DEVICE, DTYPE
    _sids9 = []
    for _pid, _seed in CFG.pilot_pairs:
        _sid = f"{scenario_id(_pid, _seed)}"
        _grab = BandLandscapeGrabber(pipe.transformer, _band_blocks, T_STAR)
        torch.manual_seed(_seed)
        gen = torch.Generator("cpu").manual_seed(_seed)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        E = EMBEDS[_pid]
        _grab.attach()
        try:
            with torch.inference_mode():
                out = pipe(prompt_embeds=E["pe"].to(device, pipe_dtype),
                       prompt_attention_mask=E["pm"].to(device),
                       negative_prompt_embeds=E["ne"].to(device, pipe_dtype),
                       negative_prompt_attention_mask=E["nm"].to(device),
                       # the default negative_prompt is "", which clashes
                       # with the precomputed negative embeds
                       negative_prompt=None,
                       height=CFG.height, width=CFG.width,
                       num_inference_steps=CFG.num_steps,
                       guidance_scale=CFG.guidance_scale,
                       generator=gen, output_type="pil",
                       use_resolution_binning=False)
        finally:
            _grab.detach()
        assert not _grab.errors, f"hook errors in {_sid}: {_grab.errors}"
        assert set(_grab.tensors) == set(_band_blocks), \
            (f"{_sid}: captured blocks {sorted(_grab.tensors)} != "
             f"band {_band_blocks}")
        assert all(_grab.counters[b] == CFG.num_steps
                   for b in _band_blocks), \
            f"{_sid}: unexpected fire counts {_grab.counters}"
        for _j, _b in enumerate(_band_blocks):
            _SUM[_j] += _grab.tensors[_b].astype(np.float32)
        _n_used9 += 1
        _sids9.append(_sid)
        print(f"[S9c] {_sid}: {len(_band_blocks)} band layers captured")
        del _grab, out

    _MEAN9 = (_SUM / max(_n_used9, 1)).astype(np.float16)
    save_npz_atomic(_bl_path, landscape=_MEAN9,
                    blocks=np.asarray(_band_blocks, np.int32),
                    meta_json=meta_to_npz_field({
                        **_want_meta, "n_used": _n_used9,
                        "pilot_sids": _sids9,
                        "precision_mode": PRECISION_MODE}))
    del _SUM
    print(f"[S9c] band landscapes saved -> {_bl_path} "
          f"(mean over {_n_used9} pilot scenarios)")


In [ ]:
# =====================================================================
# S9c figures. Token x channel landscape per band layer
# =====================================================================
# Draws the capture from the cell above as one full-size landscape per
# band layer, a grid of all band layers on one shared color scale and the
# landscape averaged over the band.

_bl_path = os.path.join(PATHS["analysis"], "band_landscapes.npz")
if not os.path.exists(_bl_path):
    print("[S9c-fig] band_landscapes.npz not found; run the S9c capture "
          "cell first (it needs the loaded pipeline)")
else:
    with np.load(_bl_path, allow_pickle=False) as _z:
        _BLK9 = [int(b) for b in _z["blocks"]]
        _BL9 = _z["landscape"].astype(np.float32)
        _m9 = meta_from_npz(_z)
    _ls9, _ts9 = int(_m9["lstar"]), int(_m9["tstar"])
    _n9 = int(_m9.get("n_used", 0))
    if _m9.get("config_hash") != CONFIG_HASH:
        print(f"[S9c-fig][WARN] capture config {_m9.get('config_hash')} "
              f"differs from current config {CONFIG_HASH}")
    _lo9f, _hi9f = int(CFG.band[0]), int(CFG.band[1])
    if [_BLK9[0], _BLK9[-1]] != [_lo9f, _hi9f] or \
            len(_BLK9) != _hi9f - _lo9f + 1:
        print(f"[S9c-fig][WARN] captured blocks {_BLK9[0]}..{_BLK9[-1]} "
              f"({len(_BLK9)}) differ from the configured band "
              f"{_lo9f}..{_hi9f}; rendering what was captured")
    assert _BL9.shape == (len(_BLK9), N_IMG, CFG.d_model), _BL9.shape

    _pool9 = max(1, CFG.d_model // 1024)
    _TP9 = 8
    _Dp9 = (CFG.d_model // _pool9) * _pool9

    def _pool_land(A):
        """Same pooling as the D1 landscape, max over channel groups of
        _pool9 and token groups of _TP9."""
        P = A[:, :_Dp9].reshape(N_IMG, _Dp9 // _pool9, _pool9).max(axis=2)
        return P.reshape(N_IMG // _TP9, _TP9, -1).max(axis=1)

    _STACK9 = np.stack([_pool_land(_BL9[j]) for j in range(len(_BLK9))])
    _pos9 = _STACK9[_STACK9 > 0]
    _vmin9 = max(float(np.percentile(_pos9, HEAT_FLOOR_PCTL)), 1e-6)
    _vmax9 = max(float(np.percentile(_STACK9, 99.9)), _vmin9 * 10.0)
    _norm9 = LogNorm(vmin=_vmin9, vmax=_vmax9)

    _prof9 = _BL9.mean(axis=(0, 1))          # band+token mean per channel
    _lab9 = sorted(set([int(c) for c in topk_ids(_prof9, 5)]
                       + (RECURRING[:10] if "RECURRING" in globals()
                          else [])))
    _pdesc = (f"tokens max-pooled in groups of {_TP9}"
              + (f" and channels by a factor of {_pool9}" if _pool9 > 1
                 else ""))

    # ---- (a) landscape averaged over the band, full size, labeled
    _BAND_MEAN9 = _pool_land(_BL9.mean(axis=0))
    fig, ax = plt.subplots(figsize=(FIG_PAGE, 3.0))
    im = ax.imshow(_BAND_MEAN9, aspect="auto", origin="lower",
                   interpolation="nearest", cmap="magma", norm=_norm9,
                   extent=[0, _Dp9, 0, N_IMG], rasterized=True)
    ax.set_xlabel("channel index"); ax.set_ylabel("image token index")
    label_channels_top(ax, _lab9, _lab9)
    fig_title(ax, f"{MODEL_LABEL}: mean $|x|$ per channel and image "
                  f"token, averaged over the high-norm band (layers "
                  f"{_BLK9[0]} to {_BLK9[-1]}, step {_ts9})", pad=22)
    cb = fig.colorbar(im, ax=ax, pad=0.01, extend="max")
    cb.set_label(r"mean $|x|$ (log scale)")
    save_fig(fig, fig_stem("D1_band_mean_landscape"), caption=(
        f"Token by channel activation landscape of {MODEL_LABEL} averaged "
        f"over the layers of the pre-registered band ({_BLK9[0]} to "
        f"{_BLK9[-1]}) at denoising step {_ts9}, mean over {_n9} pilot "
        f"scenarios. Labels above the panel mark the largest channels of "
        f"the band mean and every globally recurring channel. For "
        f"rendering, {_pdesc}; the color scale is shared with the "
        f"per-layer landscapes and clipped at the 99.9th percentile."))

    # ---- (b) grid overview, one panel per band layer, one shared scale

    _nc9 = 4
    _nr9 = int(np.ceil(len(_BLK9) / _nc9))
    fig, axes = plt.subplots(_nr9, _nc9,
                             figsize=(FIG_PAGE, 0.95 * _nr9 + 0.55),
                             sharex=True, sharey=True)
    _axf9 = np.atleast_1d(axes).flat
    for _ax, (_j, _b) in zip(_axf9, enumerate(_BLK9)):
        im = _ax.imshow(_STACK9[_j], aspect="auto", origin="lower",
                        interpolation="nearest", cmap="magma", norm=_norm9,
                        extent=[0, _Dp9, 0, N_IMG], rasterized=True)
        _ax.set_title(f"layer {_b}", fontsize=5.5, pad=2)
        _ax.tick_params(labelsize=5)
    for _ax in list(_axf9)[len(_BLK9):]:
        _ax.axis("off")
    fig.supxlabel("channel index", fontsize=_BASE_PT - 1)
    fig.supylabel("image token index", fontsize=_BASE_PT - 1)
    fig.suptitle(f"{MODEL_LABEL}: band-layer activation landscapes "
                 f"(step {_ts9}, shared color scale)",
                 fontsize=_BASE_PT + 0.5)
    cb = fig.colorbar(im, ax=np.atleast_1d(axes).ravel().tolist(),
                      pad=0.01, extend="max")
    cb.set_label(r"mean $|x|$ (log scale)", fontsize=_BASE_PT - 1)
    cb.ax.tick_params(labelsize=5.5)
    save_fig(fig, fig_stem("D1_band_layer_landscapes"), caption=(
        f"Token by channel activation landscapes of {MODEL_LABEL} at "
        f"every layer of the pre-registered band ({_BLK9[0]} to "
        f"{_BLK9[-1]}), denoising step {_ts9}, mean over {_n9} pilot "
        f"scenarios, all panels on one shared log color scale clipped at "
        f"the 99.9th percentile. A channel column that brightens or "
        f"fades across panels changes its magnitude across layers of the "
        f"band. For rendering, {_pdesc}."))

    # ---- (c) one full-size labeled landscape file per band layer
    _bl_dir = os.path.join(PATHS["figures"], "band_landscapes")
    os.makedirs(_bl_dir, exist_ok=True)
    for _j, _b in enumerate(_BLK9):
        f2, a2 = plt.subplots(figsize=(FIG_PAGE, 2.6))
        i2 = a2.imshow(_STACK9[_j], aspect="auto", origin="lower",
                       interpolation="nearest", cmap="magma", norm=_norm9,
                       extent=[0, _Dp9, 0, N_IMG], rasterized=True)
        a2.set_xlabel("channel index"); a2.set_ylabel("image token index")
        label_channels_top(a2, _lab9, _lab9)
        fig_title(a2, f"{MODEL_LABEL}: activation landscape at layer "
                      f"{_b} (step {_ts9})", pad=22)
        c2 = f2.colorbar(i2, ax=a2, pad=0.01, extend="max")
        c2.set_label(r"mean $|x|$ (log scale)")
        for _ext in ("pdf", "png"):
            f2.savefig(os.path.join(_bl_dir,
                                    f"D1_landscape_layer{_b:02d}.{_ext}"),
                       bbox_inches="tight",
                       dpi=300 if _ext == "png" else None)
        plt.close(f2)
    print(f"[S9c-fig] wrote {len(_BLK9)} per-layer landscape figures -> "
          f"{_bl_dir}")


## S10. D2, recurrence

For each channel, the recurrence f_k is the fraction of scenarios that have
it in their top k, for k = 1, 5 and 10, taken from the per-scenario top-20
lists. The union sizes |U_k| are the main result. A union of size k means
the same channels show up in every scenario. A union that keeps growing with
more scenarios means the top channels are chosen per scenario. The cell also
reports the recurring sets (f_k of at least 0.9) and the 95 percent coverage
curve.


In [ ]:
# =====================================================================
# S10. D2, recurrence of per-scenario top channels
# =====================================================================
# f_k(d) is the fraction of clean scenarios that have channel d in their
# top-k by the primary statistic, for k in {1, 5, 10}. The main number is
# |U_k|, the size of the union of all per-scenario top-k sets.
assert "CLEAN_SCENARIOS" in globals(), "run the S8 audit cell first"
DECISION = load_decision(PATHS["meta"])
L_STAR, T_STAR = int(DECISION["l_star"]), int(DECISION["t_star"])
PRIMARY_STAT = DECISION["primary_stat"]

TOP_IDS, TOP_SCORES, TOP_SIDS = collect_top_lists(
    PATHS["main"], CLEAN_SCENARIOS, CFG.save_full_tensor, L_STAR, T_STAR)
S_USED = TOP_IDS.shape[0]

# Spot check on the first clean scenario. Its stored mean|x| top-20 list
# has to match the ranking recomputed from the raw statistics in the same
# file. A mismatch means mixed decisions or stale files, so stop here.
_sid0 = CLEAN_SCENARIOS[0][0]
_p0 = scenario_paths(PATHS["main"], _sid0, CFG.save_full_tensor,
                     L_STAR, T_STAR)
with np.load(_p0["npz"], allow_pickle=False) as _z0:
    _sa0 = _z0[f"b{L_STAR:02d}__mean_abs"][T_STAR].astype(np.float64)
    _stored0 = _z0["top20_ids_mean_abs"].astype(np.int64)
assert np.array_equal(topk_ids(_sa0, CFG.top_save), _stored0), (
    f"{_sid0}: stored top-20 list does not match the statistics in the "
    f"same file. The main-run files may mix pilot decisions; delete the "
    f"affected files and re-run Phase B (S6) and the audit (S8).")
print(f"[D2] integrity: stored top-20 lists match recomputation ({_sid0})")
FREQS = recurrence_freqs(TOP_IDS, list(CFG.k_list), CFG.d_model)
UNIONS = {k: int((FREQS[k] > 0).sum()) for k in sorted(FREQS)}
G_SETS = {k: sorted(int(c) for c in
                    np.where(FREQS[k] >= CFG.recurring_set_threshold)[0])
          for k in CFG.k_list}

print(f"[D2] scenarios used: {S_USED} | primary stat: {PRIMARY_STAT}")
print(f"[D2] HEADLINE union sizes: " +
      ", ".join(f"|U_{k}|={UNIONS[k]} ({UNIONS[k]/CFG.d_model:.2%} of D)"
                for k in sorted(UNIONS)))
for k in CFG.k_list:
    print(f"[D2] globally recurring set G_{k} (f_{k} >= "
          f"{CFG.recurring_set_threshold}): {G_SETS[k]}")

# ---- recurrence figure (horizontal bars)
_u = np.where(FREQS[max(FREQS)] > 0)[0]
_order = sorted(_u, key=lambda dd: (-FREQS[10][dd], -FREQS[5][dd],
                                    -FREQS[1][dd]))
_show = _order[:15][::-1]
_y = np.arange(len(_show))
_h = 0.8 / len(CFG.k_list)
fig, ax = plt.subplots(figsize=(FIG_COL, 0.17 * len(_show) + 0.9))
for i, k in enumerate(sorted(CFG.k_list)):
    ax.barh(_y + (i - (len(CFG.k_list) - 1) / 2) * _h,
            [FREQS[k][c] for c in _show], height=_h * 0.9,
            color=OKABE_ITO[i], label=f"$k={k}$")
ax.set_yticks(_y, [str(int(c)) for c in _show])
ax.set_ylabel("channel index")
ax.set_xlabel(f"recurrence $f_k$ over {S_USED} scenarios")
ax.set_xlim(0, 1.0)
ax.axvline(CFG.recurring_set_threshold, color="0.3", ls=":", lw=0.8)
legend_above(ax, ncol=3)
fig_title(ax, f"{MODEL_LABEL}: cross-scenario recurrence\nof top-$k$ "
              f"channels (layer {L_STAR})", pad=22)
_utxt = "; ".join(f"|U_{k}| = {v} ({v / CFG.d_model:.1%} of D)"
                  for k, v in UNIONS.items())
save_fig(fig, fig_stem("D2_recurrence"), caption=(
    f"Recurrence of per-scenario top-k massive-activation channels in "
    f"{MODEL_LABEL} at block {L_STAR}, step {T_STAR}, ranked by "
    f"{PRIMARY_WORDS[PRIMARY_STAT]}. For each channel, f_k is the "
    f"fraction of the {S_USED} scenarios in which it ranks in that "
    f"scenario's top k; the dotted line marks the recurring-set threshold "
    f"of {CFG.recurring_set_threshold}. The {len(_show)} most recurrent "
    f"channels are shown; the complete table is provided as a CSV. Union "
    f"sizes across scenarios: {_utxt}."))
if len(_order) > 15:
    print(f"[D2][note] union of top-20 sets has {len(_order)} channels; "
          f"the figure shows the top 15 by f_10, full list in recurrence.csv")

# ---- coverage curve
_c, _cum, _n95 = coverage_stats(FREQS[10] * S_USED)
fig, ax = plt.subplots(figsize=(FIG_COL, 2.0))
ax.plot(np.arange(1, len(_cum) + 1), _cum, lw=1.2, color=OKABE_ITO[0])
ax.axhline(0.95, color="0.3", ls="--", lw=0.8)
ax.axvline(_n95, color=OKABE_ITO[3], ls=":", lw=1.0)
ax.set_xlim(0, max(30, _n95 * 2)); ax.set_ylim(0, 1.02)
ax.set_xlabel("channels, sorted by top-10 membership")
ax.set_ylabel("cumulative fraction of\ntop-10 memberships")
fig_title(ax, f"{MODEL_LABEL}: concentration of\ntop-10 membership",
          pad=6)
save_fig(fig, fig_stem("D2_coverage"), caption=(
    f"Concentration of top-10 membership in {MODEL_LABEL}: the {_n95} "
    f"most frequent channels account for 95 percent of all per-scenario "
    f"top-10 memberships across the {S_USED} scenarios."))

_rec_df = pd.DataFrame({
    "channel": [int(c) for c in _order],
    **{f"f{k}": [float(FREQS[k][c]) for c in _order] for k in sorted(FREQS)}})
_rec_df.to_csv(os.path.join(PATHS["analysis"], "recurrence.csv"), index=False)
print(f"[D2] recurrence.csv written ({len(_rec_df)} channels); "
      f"95% of top-10 memberships covered by {_n95} channel(s)")


## S11. D3, ordered candidate table

One row per channel in the union of the per-scenario top-20 sets. The
columns cover recurrence, score level and spread, rank stability, sign
structure, spatial uniformity and the ratio of the 99th percentile to the
maximum, which tells a spike from a wall. The last columns separate channels
that form near-constant walls over all tokens from channels driven by a few
extreme tokens.


In [ ]:
# =====================================================================
# S11. D3, ordered candidate table
# =====================================================================
# Writes a CSV with one row per channel in the union of the per-scenario
# top-20 sets. Columns cover recurrence (f1, f5, f10, f20), score level and
# spread, rank stability, sign structure, spike vs wall shape (p99/max)
# and spatial uniformity (frac_wall, token CV). The last two separate
# near-constant walls over all tokens from localized channels.

assert "CLEAN_SCENARIOS" in globals(), "run the S8 audit cell first"
DECISION = load_decision(PATHS["meta"])
L_STAR, T_STAR = int(DECISION["l_star"]), int(DECISION["t_star"])
PRIMARY_STAT = DECISION["primary_stat"]

if "FREQS" not in globals():
    _ti, _ts, _ = collect_top_lists(PATHS["main"], CLEAN_SCENARIOS,
                                    CFG.save_full_tensor, L_STAR, T_STAR)
    FREQS = recurrence_freqs(_ti, list(CFG.k_list), CFG.d_model)
if "LAND" not in globals():
    with np.load(os.path.join(PATHS["analysis"], "landscape.npz"),
                 allow_pickle=False) as _z:
        LAND = {"frac_wall_mean": _z["frac_wall_mean"].astype(np.float64),
                "cv_abs_mean": _z["cv_abs_mean"].astype(np.float64)}
    print("[D3] uniformity metrics loaded from analysis/landscape.npz")

CANDIDATES = build_candidate_table(
    CLEAN_SCENARIOS, PATHS["main"], L_STAR, T_STAR, PRIMARY_STAT,
    FREQS, LAND, CFG.dg_anchor_channels, CFG.save_full_tensor, CFG.d_model)
_csv = os.path.join(PATHS["analysis"], "candidate_channels.csv")
CANDIDATES.to_csv(_csv, index=False)
print(f"[D3] candidate table -> {_csv} ({len(CANDIDATES)} channels)")
with pd.option_context("display.width", 160, "display.max_columns", 20,
                       "display.float_format", lambda v: f"{v:.3f}"):
    print(CANDIDATES.head(20).to_string(index=False))
try:
    from IPython.display import display
    display(CANDIDATES.head(20))
except Exception:
    pass
print("\n[D3] reading guide: f10=1.0 & rank_worst<=10 -> globally recurring; "
      "sign_consistency_mean < 0.9 -> |mean|-cancellation risk; "
      "frac_wall high & cv_abs_tokens low -> DG-style wall; "
      "p99_over_max_mean << 1 -> single-token spike, not a wall.")


## S12. D4, robustness of the recurring channels

The first figure shows the magnitude of each recurring channel in every
scenario, grouped by prompt. A one-way variance decomposition splits its
variance into a between-prompt and a within-prompt part. A between-prompt
share near zero would mean the magnitude doesn't depend on the prompt. The
second figure shows the magnitude across denoising steps on a relative time
axis, as the mean and one standard deviation over scenarios.


In [ ]:
# =====================================================================
# S12. D4, robustness of the recurring channels
# =====================================================================
# D4a shows the per-scenario magnitude grouped by prompt, with a variance
# split into between-prompt and within-prompt parts. D4b shows the
# magnitude over denoising on a relative time axis, as the mean and one
# standard deviation over scenarios.
assert "CLEAN_SCENARIOS" in globals(), "run the S8 audit cell first"
DECISION = load_decision(PATHS["meta"])
L_STAR, T_STAR = int(DECISION["l_star"]), int(DECISION["t_star"])
PRIMARY_STAT = DECISION["primary_stat"]
if "FREQS" not in globals():
    _ti, _, _ = collect_top_lists(PATHS["main"], CLEAN_SCENARIOS,
                                  CFG.save_full_tensor, L_STAR, T_STAR)
    FREQS = recurrence_freqs(_ti, list(CFG.k_list), CFG.d_model)

_g = [int(c) for c in np.where(FREQS[10] >= CFG.recurring_set_threshold)[0]]
if not _g:
    _g = [int(c) for c in topk_ids(FREQS[10], 3)]
    print(f"[D4][note] no channel reaches f10 >= "
          f"{CFG.recurring_set_threshold}; showing top-3 by f10 instead")
G_CHANNELS = sorted(_g)[:8]
print(f"[D4] recurring channels analyzed: {G_CHANNELS}")

_pids = sorted({pid for _, pid, _, _ in SCENARIOS})
_seeds = list(CFG.seeds)
_pi = {p: i for i, p in enumerate(_pids)}
_si = {s: j for j, s in enumerate(_seeds)}

# ---- D4a per-scenario scores at (l*, t*), grouped by prompt
_mat = np.full((len(_pids), len(_seeds), len(G_CHANNELS)), np.nan)
for sid, pid, seed, _ in CLEAN_SCENARIOS:
    p = scenario_paths(PATHS["main"], sid, CFG.save_full_tensor,
                       L_STAR, T_STAR)
    vec, _ = scenario_score_vector(p["npz"], L_STAR, T_STAR, PRIMARY_STAT)
    _mat[_pi[pid], _si[seed]] = vec[G_CHANNELS]

fig, ax = plt.subplots(figsize=(FIG_PAGE, 2.4))
_rng = np.random.default_rng(0)
for j, ch in enumerate(G_CHANNELS):
    _col = OKABE_ITO[j % len(OKABE_ITO)]
    for i in range(len(_pids)):
        _x = i + (_rng.random(len(_seeds)) - 0.5) * 0.5
        ax.scatter(_x, _mat[i, :, j], s=5, color=_col, alpha=0.85,
                   label=f"{ch}" if i == 0 else None)
for b in range(1, len(_pids)):
    ax.axvline(b - 0.5, color="0.88", lw=0.5, zorder=0)
ax.set_xticks(range(len(_pids)), _pids, rotation=90)
ax.set_yscale("log")
ax.set_ylabel(PRIMARY_TEX[PRIMARY_STAT])
legend_above(ax, ncol=min(4, len(G_CHANNELS)), title="channel")
fig_title(ax, f"{MODEL_LABEL}: recurring-channel magnitude by prompt "
              f"(layer {L_STAR}, step {T_STAR}, "
              f"{len(_seeds)} seeds per prompt)", pad=30)

print("[D4a] prompt-vs-seed variance decomposition "
      "(eta2 near 0 means prompt-independent):")
_d4a_rows = []
for j, ch in enumerate(G_CHANNELS):
    sub = _mat[:, :, j]
    rows_ok = ~np.isnan(sub).any(axis=1)
    if rows_ok.sum() >= 2:
        vd = variance_decomposition(sub[rows_ok])
        vd.update(channel=ch, n_prompts_used=int(rows_ok.sum()))
        _d4a_rows.append(vd)
        print(f"    ch {ch:4d}: eta2_prompt={vd['eta2_prompt']:.3f}  "
              f"(SSb={vd['ss_between']:.3g}, SSw={vd['ss_within']:.3g}, "
              f"prompts used {vd['n_prompts_used']})")
pd.DataFrame(_d4a_rows).to_csv(
    os.path.join(PATHS["analysis"], "d4a_variance_decomposition.csv"),
    index=False)
_eta_txt = "; ".join(f"channel {r['channel']}: {r['eta2_prompt']:.2f}"
                     for r in _d4a_rows)
save_fig(fig, fig_stem("D4a_scores_by_prompt"), caption=(
    f"Per-scenario magnitude of the recurring channels of {MODEL_LABEL} "
    f"at block {L_STAR}, step {T_STAR}, grouped by prompt with five seeds "
    f"per prompt. Between-prompt variance shares (eta squared) from a "
    f"one-way decomposition: {_eta_txt}. A share near zero indicates "
    f"prompt independence; a share near one indicates prompt-driven "
    f"magnitude."))

# ---- D4b score vs relative denoising position

_curves = np.stack([per_step_scores(
    scenario_paths(PATHS["main"], sid, CFG.save_full_tensor,
                   L_STAR, T_STAR)["npz"],
    L_STAR, PRIMARY_STAT, G_CHANNELS) for sid, _, _, _ in CLEAN_SCENARIOS])
_mu, _sd = _curves.mean(axis=0), _curves.std(axis=0)
_x = (np.arange(CFG.num_steps) + 1) / CFG.num_steps
fig, ax = plt.subplots(figsize=(FIG_COL, 2.2))
for j, ch in enumerate(G_CHANNELS):
    _col = OKABE_ITO[j % len(OKABE_ITO)]
    ax.plot(_x, _mu[:, j], lw=1.1, color=_col, label=f"{ch}")
    ax.fill_between(_x, _mu[:, j] - _sd[:, j], _mu[:, j] + _sd[:, j],
                    color=_col, alpha=0.15, lw=0)
ax.axvline((T_STAR + 1) / CFG.num_steps, color="0.3", ls="--", lw=0.8)
ax.set_xlabel(r"relative denoising position $t/T$")
ax.set_ylabel(PRIMARY_TEX[PRIMARY_STAT])
legend_above(ax, ncol=min(4, len(G_CHANNELS)), title="channel")
fig_title(ax, f"{MODEL_LABEL}: recurring-channel\nmagnitude across "
              f"denoising (layer {L_STAR})", pad=30)
save_fig(fig, fig_stem("D4b_scores_vs_step"), caption=(
    f"Magnitude of the recurring channels of {MODEL_LABEL} at block "
    f"{L_STAR} across denoising, mean and one standard deviation over "
    f"{len(CLEAN_SCENARIOS)} scenarios. The dashed line marks the "
    f"analysis step. The horizontal axis is the relative position in the "
    f"sampling trajectory, which allows comparison between models with "
    f"different step counts."))
save_npz_atomic(os.path.join(PATHS["analysis"], "d4b_step_curves.npz"),
                mean=_mu.astype(np.float32), std=_sd.astype(np.float32),
                channels=np.asarray(G_CHANNELS, np.int32),
                meta_json=meta_to_npz_field({"l_star": L_STAR,
                                             "t_star": T_STAR,
                                             "primary_stat": PRIMARY_STAT,
                                             "n_scenarios": len(CLEAN_SCENARIOS)}))
print("[D4] complete")


## S13. Run manifest

Writes one JSON file with what is needed to audit or extend this run. It
holds library versions, the GPU, the precision mode, the config and its
hash, the full pilot decision with its evidence, completion counts, the main
numbers, caveats and the storage used per folder. Later analyses that
combine scenarios and channels can read the per-scenario npz files directly
without generating anything again.


In [ ]:
# =====================================================================
# S13. Run manifest and summary
# =====================================================================
# Writes meta/run_manifest.json.

assert "CLEAN_SCENARIOS" in globals(), "run the S8 audit cell first"
DECISION = load_decision(PATHS["meta"])
L_STAR, T_STAR = int(DECISION["l_star"]), int(DECISION["t_star"])
PRIMARY_STAT = DECISION["primary_stat"]
if "FREQS" not in globals():
    _ti, _, _ = collect_top_lists(PATHS["main"], CLEAN_SCENARIOS,
                                  CFG.save_full_tensor, L_STAR, T_STAR)
    FREQS = recurrence_freqs(_ti, list(CFG.k_list), CFG.d_model)
UNIONS = {int(k): int((FREQS[k] > 0).sum()) for k in sorted(FREQS)}
G10 = [int(c) for c in np.where(FREQS[10] >= CFG.recurring_set_threshold)[0]]

def _dir_gib(d):
    return sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(d) for f in fs) / 2**30

_caveats = []
if PRECISION_MODE != "bf16_full":
    _caveats.append(
        f"precision_mode={PRECISION_MODE}: activations were measured under "
        f"reduced precision or CPU offloading; channel identities are "
        f"expected to be stable, magnitudes may shift slightly relative to "
        f"bf16.")
if not DECISION["evidence"]["stat_gate"]["passed"]:
    _caveats.append(
        "P4 statistic gate FAILED in the pilot: the primary statistic is "
        "mean|x|, so top-k lists are NOT directly comparable to the Eq. 1 "
        "ranking of Turri et al. (|mean|). Mixed-sign hits are listed in "
        "pilot_decision.json.")
if not DECISION["evidence"]["anchor"]["found"]:
    _caveats.append("The bottom-right position anchor was not reproduced in "
                    "the pilot; run proceeded under allow_missing_anchor "
                    "override.")

_cand_csv = os.path.join(PATHS["analysis"], "candidate_channels.csv")
_top_rows = []
if os.path.exists(_cand_csv):
    _top_rows = pd.read_csv(_cand_csv).head(10)[
        ["channel", "f10", "score_mean", "sign_consistency_mean",
         "frac_wall"]].to_dict("records")

MANIFEST = {
    "experiment": "PixArt-Sigma massive-activation channel recurrence "
                  "(landscape + recurrence deliverables)",
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "gpu": {"name": GPU_NAME, "total_gib": round(TOTAL_GIB, 1),
            "compute_capability": list(CC)},
    "precision_mode": PRECISION_MODE, "model_dtype": str(DTYPE),
    "lib_versions": LIB_VERSIONS,
    "config": json.loads(json.dumps(asdict(CFG), default=str)),
    "config_hash": CONFIG_HASH,
    "prompt_manifest": {"source": PROMPT_MANIFEST["source"],
                        "sha256": PROMPT_MANIFEST["sha256"]},
    "decision": DECISION,
    "completion": {"scenarios_total": len(SCENARIOS),
                   "scenarios_clean": len(CLEAN_SCENARIOS)},
    "headline": {"l_star": L_STAR, "t_star": T_STAR,
                 "primary_stat": PRIMARY_STAT,
                 "union_sizes": UNIONS,
                 "recurring_set_G10": G10,
                 "top_candidates": _top_rows},
    "caveats": _caveats,
    "artifacts": {
        "pilot_decision": os.path.join(PATHS["meta"], "pilot_decision.json"),
        "prompts": os.path.join(PATHS["meta"], "prompts.json"),
        "scenarios": os.path.join(PATHS["meta"], "scenarios.csv"),
        "run_log": RUN_LOG_PATH, "audit": os.path.join(PATHS["meta"], "audit.csv"),
        "landscape": os.path.join(PATHS["analysis"], "landscape.npz"),
        "recurrence": os.path.join(PATHS["analysis"], "recurrence.csv"),
        "candidates": _cand_csv, "figures_dir": PATHS["figures"],
        "per_scenario_dir": PATHS["main"]},
    "storage_gib": {k: round(_dir_gib(PATHS[k]), 2)
                    for k in ("pilot", "main", "figures", "analysis", "embeds")},
}
save_json_atomic(MANIFEST, os.path.join(PATHS["meta"], "run_manifest.json"))

print("=" * 70); print("RUN MANIFEST / HANDOFF SUMMARY"); print("=" * 70)
print(f"  GPU {GPU_NAME} | mode {PRECISION_MODE} | config {CONFIG_HASH}")
print(f"  decision: l*={L_STAR}  t*={T_STAR}  primary={PRIMARY_STAT}")
print(f"  clean scenarios: {len(CLEAN_SCENARIOS)}/{len(SCENARIOS)}")
print(f"  HEADLINE union sizes: " +
      ", ".join(f"|U_{k}|={v}" for k, v in UNIONS.items()))
print(f"  recurring set G10 (f10>={CFG.recurring_set_threshold}): {G10}")
for c in _caveats:
    print(f"  CAVEAT: {c}")
print(f"  storage: " + ", ".join(f"{k}={v} GiB"
      for k, v in MANIFEST["storage_gib"].items()))
print(f"  -> {os.path.join(PATHS['meta'], 'run_manifest.json')}")
print("=" * 70)
print("Handoff = meta/ + analysis/ + figures/ + per-scenario npz in main/ "
      "(downstream combined scenario x channel analyses consume the npz "
      "files directly; per-token norms at l* are already logged for "
      "token-level follow-up analyses).")
